### Universidade Federal da Bahia
### Escola Politécnica
### Programa de Pós-Graduação em Engenharia Industrial
### Discente: Ghabriel Anton Gomes de Sá
### Orientadores: Marcelo Embiruçu e Cristiano Fontes
**Ultima Modificação: 23/11/2025**

<center><h2>UMA ABORDAGEM COMBINANDO UMA ESTIMATIVA DE PESOS INICIAIS BASEADA EM LINEARIZAÇÃO E ALGORITMO CONSTRUTIVISTA PARA A CRIAÇÃO DE REDES NEURAIS DE PROPAGAÇÃO DIRETA COM UMA ÚNICA CAMADA OCULTA APLICADAS À IDENTIFIÇÃO DE MODELOS COM MÚLTIPLAS ENTRADAS E ÚNICA SAÍDA EM PROBLEMAS DE REGRESSÃO COM POSSIBILIDADE DE RESTRIÇÕES NOS SINAIS DOS GANHOS</center></h2>

##### <p style='text-align: justify;'>O Método para a obtenção dos pesos iniciais propõe que: (i) seja realizada a linearização, via série de Taylor, do modelo não-linear a ser utilizado para a modelagem do conjunto de dados (o modelo consiste em uma rede neural com p entradas e um neurônio na camada de saída), onde o ponto de equilíbrio para a linearização é a média das suas respectivas variáveis (x,y); (ii) seja realizada a Regressão Linear Múltipla, via Método dos Mínimos Quadrados, do conjunto de dados a ser modelado; (iii) através da comparação matemática entre (i) e (ii), obtenha-se os pesos que conectam os neurônios de entrada ao primeiro neurônio oculto da rede. </p>

##### <p style='text-align: justify;'>Em seguida, a rede continua a aumentar o número de neurônios ocultos a partir de uma abordagem construtivista. Toda a rede é treinada a cada adição de um novo neurônio oculto, sendo que os pesos obtidos no final de um treinamento são utilizados como pesos iniciais para o treinamento subsequente. Os pesos iniciais do neurônio oculto a ser adicionado são obtidos a partir do Método de Xavier, enquanto biases são inicializados com valor nulo. Comparou-se o método proposto com outros dois métodos (RIXM e ELM) sem e com restrições nos sinais dos ganhos. </p>

## Importação de pacotes

In [ ]:
# json: para leitura de arquivos de configuração
# datetime: para contagem de tempo
# pathlib: para manipulação de arquivos e diretórios
# copy: para cópia de objetos
# logging: para registro de logs
# pickle: para serialização de objetos
import json
from datetime import datetime
from pathlib import Path
import copy
import logging
import pickle

# pandas: para manipulação de dados
import pandas as pd

# MinMaxScaler: para normalização de dados
# train_test_split: para divisão de dados em treino e teste
# r2_score e mean_squared_error: para cálculo de métricas
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error as mse

# statsmodels: para resolução de sistemas de equações não-lineares
# scipy.optimize: para resolução de sistemas de equações não-lineares
import statsmodels.api as sm
from scipy.optimize import fsolve
from scipy.optimize import minimize

# numpy: para cálculo de álgebra linear
# math: para cálculo de funções matemáticas
# scipy.linalg: para cálculo de álgebra linear
import numpy as np
import math
from scipy.linalg import pinv

# Importação de pacotes para contagem de tempo
import time

# Importação de pacotes para criação e plotagem de gráficos
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

## CONFIG

In [ ]:
### Salvar/Carregar Config do Dataset (Apenas UMA VEZ por dataset)

def save_dataset_config(dataset_name, filename, expected_signals, data_shape, 
                        scaler_type='MinMaxScaler', base_dir=".."):
    """
    Salva configuração FIXA do dataset (não muda entre execuções).
    
    Args:
        dataset_name: Nome padronizado do dataset
        filename: Nome do arquivo original
        expected_signals: Lista de sinais esperados
        data_shape: (n_samples, n_features)
        scaler_type: Tipo de normalização usado
        base_dir: Diretório base do projeto
    """
    config_dir = Path(base_dir) / "data" / "processed" / dataset_name
    config_dir.mkdir(parents=True, exist_ok=True)
    
    config_path = config_dir / "dataset_config.json"
    
    # Verificar se já existe
    if config_path.exists():
        print(f"ℹ️  Config já existe: {config_path.relative_to(base_dir)}")
        with open(config_path, 'r') as f:
            existing_config = json.load(f)
        
        # Perguntar se quer sobrescrever
        overwrite = input("   Sobrescrever? (s/n): ")
        if overwrite.lower() != 's':
            print("   ✓ Mantendo config existente")
            return config_path
    
    # Criar config
    config = {
        'dataset_info': {
            'name': dataset_name,
            'original_filename': filename,
            'n_samples': int(data_shape[0]),
            'n_features': int(data_shape[1]),
            'n_inputs': int(data_shape[1] - 1),
            'n_outputs': 1
        },
        'expected_signals': {
            'values': expected_signals,
            'description': {
                '1': 'ganho positivo esperado',
                '-1': 'ganho negativo esperado',
                '0': 'ganho desconhecido'
            }
        },
        'preprocessing': {
            'scaler': scaler_type,
            'range': [0, 1] if scaler_type == 'MinMaxScaler' else None,
            'random_state': SEED,
            'test_size': 0.2
        },
        'metadata': {
            'created_at': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'source': 'UCI Machine Learning Repository',
            'license': 'CC BY 4.0'
        }
    }
    
    # Salvar
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Dataset config salvo: {config_path.relative_to(base_dir)}")
    return config_path


def load_dataset_config(dataset_name, base_dir=".."):
    """
    Carrega configuração do dataset.
    
    Args:
        dataset_name: Nome do dataset
        base_dir: Diretório base
    
    Returns:
        Dicionário com config ou None se não existir
    """
    config_path = Path(base_dir) / "data" / "processed" / dataset_name / "dataset_config.json"
    
    if not config_path.exists():
        print(f"⚠️ Config não encontrado: {config_path}")
        return None
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    print(f"✅ Dataset config carregado: {dataset_name}")
    return config

In [ ]:
### Salvar Dados Processados do Dataset

def save_processed_data(dataset_name, train_inputs, train_targets, 
                        test_inputs, test_targets, scaler, base_dir=".."):
    """
    Salva dados processados do dataset (split e scaler).
    Sobrescreve versão anterior se existir.
    
    Args:
        dataset_name: Nome do dataset
        train_inputs, train_targets: Dados de treino
        test_inputs, test_targets: Dados de teste
        scaler: Objeto scaler (MinMaxScaler ou StandardScaler)
        base_dir: Diretório base
    """
    data_dir = Path(base_dir) / "data" / "processed" / dataset_name
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # Salvar split
    split_path = data_dir / "train_test_split.npz"
    np.savez_compressed(
        split_path,
        train_inputs=train_inputs,
        train_targets=train_targets,
        test_inputs=test_inputs,
        test_targets=test_targets
    )
    print(f"✅ Train/test split salvo: {split_path.relative_to(base_dir)}")
    
    # Salvar scaler
    scaler_path = data_dir / "scaler_params.pkl"
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"✅ Scaler params salvo: {scaler_path.relative_to(base_dir)}")
    
    return split_path, scaler_path


def load_processed_data(dataset_name, base_dir=".."):
    """
    Carrega dados processados do dataset.
    
    Args:
        dataset_name: Nome do dataset
        base_dir: Diretório base
    
    Returns:
        Tuple: (train_inputs, train_targets, test_inputs, test_targets, scaler)
    """
    data_dir = Path(base_dir) / "data" / "processed" / dataset_name
    
    # Carregar split
    split_path = data_dir / "train_test_split.npz"
    if not split_path.exists():
        raise FileNotFoundError(f"Split não encontrado: {split_path}")
    
    data = np.load(split_path)
    train_inputs = data['train_inputs']
    train_targets = data['train_targets']
    test_inputs = data['test_inputs']
    test_targets = data['test_targets']
    
    # Carregar scaler
    scaler_path = data_dir / "scaler_params.pkl"
    if not scaler_path.exists():
        raise FileNotFoundError(f"Scaler não encontrado: {scaler_path}")
    
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)
    
    print(f"✅ Dados processados carregados: {dataset_name}")
    return train_inputs, train_targets, test_inputs, test_targets, scaler

## Fixação de seed
A fixação de seed garante a reprodutibilidade dos resultados.

In [ ]:
SEED = 0
np.random.seed(SEED)

## Importação do banco de dados
<p style='text-align: justify;'>Para que a importação do banco de dados seja feita da maneira correta, é necessário que o banco de dados a ser importado esteja de acordo com o padrão apresentado a seguir e também que ele esteja localizado na mesma pasta que esse código. A princípio, esse código está implementado em um arquivo .ipynb.</p>

O banco de dados deve apresentar, obrigatoriamente, as seguintes características:

-Deve ser um arquivo do tipo .csv;

-Os dados devem estar delimitados por ponto e vírgula (;), ou seja, devem estar presentes em diferentes células da planilha Excel;

-As variáveis de entradas são todas as colunas, exceto a última (que trata-se da variável de saída);

-Não deveve possuir cabeçalho (primeira linha contendo nome dos atributos).

In [ ]:
# Solicita nome do arquivo (.csv) que armazena o conjunto de dados
filename = input("Digite o nome do arquivo .csv que contém o banco de dados: ")

# Realiza a importação do conjunto de dados em formato de DataFrame
try:
    data_dir = Path("../data/raw")  # Conforme estrutura do projeto
    data_path = data_dir / f"{filename}.csv"
    
    # ✅ CORRIGIDO: Usar data_path
    data = pd.read_csv(data_path, header=None, sep=';')
    
    if data.shape[1] < 2:
        raise ValueError(f"Dataset deve ter pelo menos 2 colunas (1 entrada + 1 saída). Encontrado: {data.shape[1]}")
    
    print(f"✅ Dados carregados: {data.shape[0]} amostras, {data.shape[1]-1} variáveis de entrada, 1 saída")
    print(data.head())
    
except FileNotFoundError:
    print(f"❌ Erro: O arquivo '{data_path}' não foi encontrado. Verifique o nome e a localização do arquivo.")

### Coleta dos sinais de ganhos esperados

In [ ]:
# Dicionário com sinais teóricos esperados (Tabelas 2-4 do artigo)
EXPECTED_SIGNALS_THEORY = {
    'computer_hardware': [-1, 1, 1, 1, 1, 1],
    'fish_toxicity': [-1, 1, -1, 1, 1, 1],
    'aquatic_toxicity': [1, -1, 1, 1, 1, -1, 1, 1]
}

# Mapeamento de nomes de arquivo para dataset_name
DATASET_MAPPING = {
    'machine': 'computer_hardware',
    'machine.csv': 'computer_hardware',
    'computer_hardware': 'computer_hardware',
    'qsar_fish_toxicity': 'fish_toxicity',
    'qsar_fish_toxicity.csv': 'fish_toxicity',
    'fish_toxicity': 'fish_toxicity',
    'fish': 'fish_toxicity',
    'qsar_aquatic_toxicity': 'aquatic_toxicity',
    'qsar_aquatic_toxicity.csv': 'aquatic_toxicity',
    'aquatic_toxicity': 'aquatic_toxicity',
    'aquatic': 'aquatic_toxicity'
}

def detect_dataset_name(filename):
    """
    Detecta automaticamente o nome do dataset a partir do filename.
    
    Args:
        filename: Nome do arquivo (com ou sem extensão)
    
    Returns:
        dataset_name padronizado ou None se não reconhecido
    """
    # Normalizar filename (lowercase, remover extensão)
    filename_normalized = filename.lower().replace('.csv', '')
    
    # Buscar no mapeamento
    if filename_normalized in DATASET_MAPPING:
        return DATASET_MAPPING[filename_normalized]
    
    # Buscar por substring (mais flexível)
    for key, value in DATASET_MAPPING.items():
        if key in filename_normalized or filename_normalized in key:
            return value
    
    return None

def get_expected_signals(n_inputs, filename, auto_load=True, allow_manual=False):
    """
    Obtém sinais esperados dos ganhos de forma automática.
    
    Args:
        n_inputs: Número de variáveis de entrada
        filename: Nome do arquivo de dados
        auto_load: Se True, tenta carregar automaticamente
        allow_manual: Se True, permite input manual se não reconhecer dataset
    
    Returns:
        Lista de sinais esperados e nome do dataset
    """
    dataset_name = None
    
    if auto_load:
        # Tentar detectar automaticamente
        dataset_name = detect_dataset_name(filename)
        
        if dataset_name and dataset_name in EXPECTED_SIGNALS_THEORY:
            signals = EXPECTED_SIGNALS_THEORY[dataset_name]
            
            # Validar que número de inputs está correto
            if len(signals) == n_inputs:
                print(f"✅ Dataset identificado: '{dataset_name}'")
                print(f"✅ Sinais carregados automaticamente:")
                print(f"   {signals}")
                return signals, dataset_name
            else:
                print(f"⚠️ Dataset '{dataset_name}' reconhecido, mas número de inputs não confere:")
                print(f"   Esperado: {len(signals)} inputs")
                print(f"   Encontrado: {n_inputs} inputs")
        else:
            print(f"⚠️ Dataset não reconhecido automaticamente: '{filename}'")
            print(f"   Datasets disponíveis: {list(EXPECTED_SIGNALS_THEORY.keys())}")
    
    # Se chegou aqui, não conseguiu carregar automaticamente
    if allow_manual:
        print("\n🔧 Modo manual ativado.")
        
        # Tentar coletar dataset_name manualmente
        print("\nDatasets disponíveis:")
        for i, name in enumerate(EXPECTED_SIGNALS_THEORY.keys(), 1):
            print(f"  {i}. {name} ({len(EXPECTED_SIGNALS_THEORY[name])} inputs)")
        
        choice = input("\nEscolha o dataset (número) ou deixe em branco para input manual: ")
        
        if choice.strip().isdigit():
            idx = int(choice) - 1
            dataset_names = list(EXPECTED_SIGNALS_THEORY.keys())
            if 0 <= idx < len(dataset_names):
                dataset_name = dataset_names[idx]
                signals = EXPECTED_SIGNALS_THEORY[dataset_name]
                
                if len(signals) == n_inputs:
                    print(f"✅ Usando sinais de '{dataset_name}': {signals}")
                    return signals, dataset_name
                else:
                    print(f"❌ Erro: Dataset tem {len(signals)} inputs, mas arquivo tem {n_inputs}")
        
        # Input manual completo
        print("\nDigite os sinais esperados manualmente:")
        print("  1 = ganho positivo")
        print(" -1 = ganho negativo")
        print("  0 = ganho desconhecido")
        
        expected_signals = []
        for i in range(n_inputs):
            while True:
                signal = input(f"Sinal esperado para variável {i+1}/{n_inputs}: ")
                if signal in ["1", "-1", "0"]:
                    expected_signals.append(int(signal))
                    break
                else:
                    print("❌ Entrada inválida. Use: 1, -1 ou 0")
        
        return expected_signals, None
    
    else:
        # Modo estrito: falha se não reconhecer
        raise ValueError(
            f"❌ Dataset '{filename}' não foi reconhecido automaticamente.\n"
            f"   Datasets válidos: {list(EXPECTED_SIGNALS_THEORY.keys())}\n"
            f"   Dica: Renomeie o arquivo para um dos nomes reconhecidos ou "
            f"use allow_manual=True"
        )

In [ ]:
# === EXECUÇÃO ===
# Atribui valor à variavel que reflete o número de atributos do conjunto de dados
n_inputs = data.shape[1] - 1

# Obtém sinais esperados automaticamente
try:
    expected_signals, dataset_name = get_expected_signals(
        n_inputs=n_inputs,
        filename=filename,
        auto_load=True,
        allow_manual=False  # ✅ False = Totalmente automático (falha se não reconhecer)
    )
except ValueError as e:
    print(str(e))
    print("\n🔧 Ativando modo manual...")
    expected_signals, dataset_name = get_expected_signals(
        n_inputs=n_inputs,
        filename=filename,
        auto_load=True,
        allow_manual=True  # Fallback para manual
    )

# Definir dataset_name genérico se ainda for None
if dataset_name is None:
    dataset_name = filename.lower().replace('.csv', '')
    print(f"⚠️ Usando nome genérico de dataset: '{dataset_name}'")

# Imprime resumo
print("\n" + "=" * 70)
print(f"CONFIGURAÇÃO DO EXPERIMENTO")
print("=" * 70)
print(f"Dataset: {dataset_name}")
print(f"Arquivo: {filename}.csv")
print(f"Amostras: {data.shape[0]}")
print(f"Inputs: {n_inputs}")
print(f"Sinais esperados: {expected_signals}")
print("=" * 70)

## Pré-processamento de dados

#### Obtendo primeiras informações sobre o conjunto de dados:

In [ ]:
# Imprime cinco primeiras linhas do conjunto de dados
print("Cinco primeiras linhas do conjunto de dados:")
print(data.head())

In [ ]:
# Imprime informações sobre o conjunto de dados
print("\nInformações sobre o conjunto de dados:")
print(data.info())

In [ ]:
# Imprime uma breve descrição estatística dos dados
print("\nDescrição estatística dos dados:")
print(data.describe())

#### Lidando com os dados faltantes:

In [ ]:
# Verifica a existência de dados faltantes
print("\nVerificação de dados faltantes:")
print(data.isna().sum())

In [ ]:
# Deleta dados faltantes:
data.dropna(inplace=True)

In [ ]:
# Após dropna, verificar se alguma linha foi removida
n_before = len(data)
data.dropna(inplace=True)
n_after = len(data)
if n_before > n_after:
    print(f"⚠️ {n_before - n_after} linhas removidas (continham missing values)")
else:
    print("Nenhuma linha removida (sem missing values)")

In [ ]:
# Missing values também podem estar representadas através do número 0. Por conta disso, deve-se verificar se os zeros são missing values ou se são
# inerentes ao atributo:
num_zeros = (data == 0).sum()
print('Quantidade de valores nulos:\n', num_zeros)

#### Reduzindo a escala dos dados:
Essa etapa garante que os dados estejam entre 0 e 1.

In [ ]:
# Cria uma array a partir dos dados
z = data.values

# Normaliza os dados
min_max_scaler = MinMaxScaler()
data_normalized = min_max_scaler.fit_transform(z)
print("\nDados normalizados (primeiras linhas):\n", data_normalized[:5])

#### Delimitando entrada e saída:

In [ ]:
# Delimita entradas (todas as colunas, exceto a última) e saída (última coluna)
inputs_all = data_normalized[:, :-1]
targets_all = data_normalized[:, -1]

In [ ]:
# Exibe entrada e saída e número de atributos (variáveis de entrada)
print("\nVetores de Entrada (primeiras linhas):\n", inputs_all[:5])
print("\nValores de Saída (primeiras linhas):\n", targets_all[:5])
print("\nNúmero de atributos (entradas):", n_inputs)

#### Dividindo conjunto de dados em treinamento e teste

In [ ]:
# Embaralha e divide os dados utilizando train_test_split
train_inputs, test_inputs, train_targets, test_targets = train_test_split(
    inputs_all, targets_all, test_size=0.2, random_state=SEED)  # random_state fixado garante reprodutibilidade dos experimentos

In [ ]:
# Exibe o formato dos conjuntos de treinamento e teste
print("\nFormato dos dados de treinamento e teste:")
print("train_inputs:", train_inputs.shape)
print("test_inputs:", test_inputs.shape)
print("train_targets:", train_targets.shape)
print("test_targets:", test_targets.shape)

### Salvando configurações do dataset

In [ ]:
# === SALVAR CONFIG DO DATASET (UMA VEZ) ===
print("\n📁 Salvando configuração do dataset...")

# Salvar config fixo
save_dataset_config(
    dataset_name=dataset_name,
    filename=filename,
    expected_signals=expected_signals,
    data_shape=data.shape,
    scaler_type='MinMaxScaler'
)

# Salvar dados processados (split + scaler)
save_processed_data(
    dataset_name=dataset_name,
    train_inputs=train_inputs,
    train_targets=train_targets,
    test_inputs=test_inputs,
    test_targets=test_targets,
    scaler=min_max_scaler
)

print("=" * 70)

## Método 1 - WILCAR
Modelo SFNN, treinado via Algoritmo de Retropropagação, considerando os pesos iniciais (e bias) do primeiro neurônio oculto obtidos a partir de Linearização e Análise via Série de Taylor e pesos dos demais neurônios ocultos, bem como do neurônio de saída obtidos a partir de uma inicialização randômica via Método de Xavier, preservando os pesos dos neurônios já aprendidos e utilizando-os como estimativas iniciais para o próximo treinamento.

### Obtenção dos Pesos Iniciais
Obtenção dos pesos iniciais entre os neurônios de entrada e o primeiro neurônio oculto.

In [ ]:
# Adiciona a coluna das constantes
inputs_all_cte = sm.add_constant(inputs_all, prepend=True)
res = sm.OLS(targets_all, inputs_all_cte).fit()  # Cria e ajusta o modelo
print(res.summary())  # Imprime os resultados

In [ ]:
# Definindo constantes
x_M = [np.mean(inputs_all[:, i]) for i in range(n_inputs)]
k = [res.params[i + 1] for i in range(n_inputs)]
b = res.params[0]

In [ ]:
# Definindo chutes iniciais para beta e wi como valores aleatórios entre 0 e 1:
x0 = np.random.rand(n_inputs+1, 1)

In [ ]:
# Definindo função sigmoide e sua derivada:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def dsigmoid(x):
    z = sigmoid(x)
    return z * (1 - z)

In [ ]:
# Inicializando variáveis wi e beta:
w = [0 for i in range(n_inputs)]
beta = 0

In [ ]:
# Definindo sistema de equações a ser resolvido:
def f(x):
    w, beta = x[:-1], x[-1]
    
    # Calcular UMA VEZ e reusar
    z = np.dot(w, x_M) + beta
    sig = sigmoid(z)
    sig_prime = sig * (1 - sig)  # σ'(z)
    
    # Equação 1: Intercepto
    f = [sig - sig_prime * np.dot(w, x_M) - b]
    
    # Equações 2 a n+1: Coeficientes
    for i in range(n_inputs):
        f.append(sig_prime * w[i] - k[i])
    
    return f

In [ ]:
# Resolvendo o sistema e imprimindo resultados (pesos e bias iniciais):
# Tentar múltiplos chutes iniciais
best_result = None
best_residual = float('inf')

for trial in range(5):  # 5 tentativas com chutes diferentes
    # Chute inicial baseado nos coeficientes da regressão
    if trial == 0:
        # Chute inteligente: usar os próprios coeficientes k como aproximação inicial
        x0 = np.append(k, b).reshape(-1, 1)
    else:
        # Chutes aleatórios variados
        x0 = np.random.randn(n_inputs+1, 1) * 0.5  # Normal ao invés de uniforme
    
    result, info, ier, msg = fsolve(f, x0, full_output=True)
    
    # Verificar convergência
    residual = np.linalg.norm(info['fvec'])  # Norma do vetor de resíduos
    
    if ier == 1 and residual < best_residual:  # ier=1 significa sucesso
        best_result = result
        best_residual = residual

if best_residual > 1e-6:
    print(f"⚠️ Aviso: Convergência pode não ser ótima (resíduo = {best_residual:.2e})")
else:
    print(f"✅ Convergência bem-sucedida (resíduo = {best_residual:.2e})")

print("Pesos e bias iniciais:", best_result)
result = best_result  # Usar a melhor solução encontrada

### Avaliação dos Sinais dos Ganhos do Modelo Inicial
O modelo inicial corresponde a um Perceptron.

In [ ]:
def calculate_and_compare_gains(adjusted_weights, x_base, expected_signals, n_inputs):
    """
    Calcula os sinais dos ganhos com base nos pesos ajustados e os compara com os sinais esperados.

    Parâmetros:
    - adjusted_weights: array-like
        Pesos ajustados, incluindo os pesos dos neurônios de entrada e o bias.
    - x_base: array-like
        Vetor de entrada base utilizado para calcular os ganhos.
    - expected_signals: list
        Lista contendo os sinais esperados para os ganhos de cada variável de entrada.
    - n_inputs: int
        Número de variáveis de entrada.

    Retorna:
    - None
        A função imprime os resultados da comparação entre os ganhos calculados e os sinais esperados.
    """
    w, beta = adjusted_weights[:-1], adjusted_weights[-1]
    y_base = sigmoid(np.dot(w, x_base) + beta)
    
    calculated_signals = []
    for i in range(n_inputs):
        x_var = x_base.copy()
        x_var[i] += 0.1
        new_y = sigmoid(np.dot(w, x_var) + beta)
        gain_signal = np.sign((new_y - y_base) / 0.1)
        calculated_signals.append(gain_signal)

    # Comparando os sinais dos ganhos calculados com os esperados
    for i, (calculated, expected) in enumerate(zip(calculated_signals, expected_signals)):
        print(f"Variável {i + 1}: Ganho calculado = {calculated}, Ganho esperado = {expected}")
        if expected != 0 and calculated != expected:
            print(f"  -> Divergência detectada na variável {i + 1}.")
        elif expected == 0:
            print(f"  -> Ganho desconhecido esperado; ganho calculado foi {calculated}.")
        else:
            print(f"  -> Correspondência encontrada.")

In [ ]:
# Checagem dos valores dos ganhos associados aos pesos iniciais (utiliza como referência-se a média do conjunto de teste)
x_base = np.mean(test_inputs, axis=0).reshape(-1, 1)
calculate_and_compare_gains(result, x_base, expected_signals, n_inputs)

### Definição da Classe do Método 1

In [ ]:
class Method1:
    """
    Classe para o Método 1 (WILCAR) que treina uma rede neural feedforward com uma única camada oculta.
    
    Parâmetros:
    - x: array-like
        Dados de entrada.
    - y: array-like
        Dados de saída.
    - n: int
        Número de neurônios na camada oculta.
    - previous_model: Method1, opcional
        Modelo anterior para expandir a rede neural.
    - initial_weights: array-like, opcional
        Pesos iniciais fornecidos para o primeiro neurônio oculto.
    """
    
    def __init__(self, x, y, n, previous_model=None, initial_weights=None):
        self.x, self.y = x, y.reshape(1, -1)
        self.n = n
        self.param = self.initialize_parameters(n, initial_weights) if previous_model is None else self.expand_network(previous_model, n)
        self.dims = [x.shape[0], n, 1]
        self.lr = 0.1  # Taxa de aprendizado
        self.sam = y.shape[0]  # Número de amostras

    def initialize_parameters(self, n, initial_weights):
        """
        Inicializa os parâmetros (pesos e biases) da rede neural.
        
        Parâmetros:
        - n: int
            Número de neurônios na camada oculta.
        - initial_weights: array-like, opcional
            Pesos iniciais fornecidos para o primeiro neurônio oculto.
        
        Retorna:
        - param: dict
            Dicionário contendo os pesos e biases iniciais.
        """
        param = {'W1': np.random.randn(n, self.x.shape[0]) * np.sqrt(1. / self.x.shape[0]),
                 'b1': np.zeros((n, 1)),
                 'W2': np.random.randn(1, n) * np.sqrt(1. / n),
                 'b2': np.zeros((1, 1))}
        if initial_weights is not None:
            param['W1'][0, :] = initial_weights[:-1]
            param['b1'][0, :] = initial_weights[-1]
        return param

    def expand_network(self, previous_model, n):
        """
        Expande a rede neural existente com mais neurônios na camada oculta.
        
        Parâmetros:
        - previous_model: Method1
            Modelo anterior para expandir.
        - n: int
            Número de neurônios na nova camada oculta.
        
        Retorna:
        - new_param: dict
            Dicionário contendo os novos pesos e biases expandidos.
        """
        new_param = copy.deepcopy(previous_model.param)
        if n > previous_model.n:
            additional_weights = np.random.randn(1, self.x.shape[0]) * np.sqrt(1. / self.x.shape[0])
            new_param['W1'] = np.vstack([new_param['W1'], additional_weights])
            new_param['b1'] = np.vstack([new_param['b1'], np.zeros((1, 1))])
            new_param['W2'] = np.hstack([new_param['W2'], np.random.randn(1, 1) * np.sqrt(1. / n)])
        return new_param

    def sigmoid(self, Z):
        """
        Função de ativação sigmoide.
        
        Parâmetros:
        - Z: array-like
            Entrada para a função sigmoide.
        
        Retorna:
        - array-like
            Saída da função sigmoide.
        """
        return 1 / (1 + np.exp(-Z))

    def forward(self):
        self.Z1 = self.param['W1'].dot(self.x) + self.param['b1']
        self.A1 = self.sigmoid(self.Z1)
        self.Z2 = self.param['W2'].dot(self.A1) + self.param['b2']
        self.A2 = self.sigmoid(self.Z2)
        return self.A2

    def backward(self, Yh):
        m = self.y.shape[1]
        
        # Camada de saída
        dZ2 = Yh - self.y
        dW2 = dZ2.dot(self.A1.T) / m  # ✅ Usa A1 armazenado
        db2 = np.sum(dZ2, axis=1, keepdims=True) / m
        
        # Camada oculta
        dA1 = self.param['W2'].T.dot(dZ2)
        dZ1 = dA1 * self.A1 * (1 - self.A1)  # ✅ Usa A1 armazenado
        dW1 = dZ1.dot(self.x.T) / m
        db1 = np.sum(dZ1, axis=1, keepdims=True) / m
        
        # Atualização
        self.param['W1'] -= self.lr * dW1
        self.param['b1'] -= self.lr * db1
        self.param['W2'] -= self.lr * dW2
        self.param['b2'] -= self.lr * db2

    def train_test(self, iter=2501, patience=250, tol=1e-6):
        best_loss = float('inf')
        no_improve_count = 0
        
        for i in range(iter):
            Yh = self.forward()
            current_loss = np.mean((Yh - self.y)**2)
            
            # Early stopping
            if current_loss < best_loss - tol:
                best_loss = current_loss
                no_improve_count = 0
            else:
                no_improve_count += 1
            
            if no_improve_count >= patience:
                print(f"⏹️ Early stopping na época {i}")
                break
            
            self.backward(Yh)

        # Cálculo das métricas para os dados de treinamento
        Yh_train = self.forward()
        mse_train = mse(self.y.flatten(), Yh_train.flatten())
        rmse_train = np.sqrt(mse_train) 
        r2_train = r2_score(self.y.flatten(), Yh_train.flatten())

        # Cálculo das métricas para os dados de teste
        Yh_test = self.predict(test_inputs.T)
        mse_test = mse(test_targets.T, Yh_test.flatten())
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(test_targets.T, Yh_test.flatten())

        return mse_train, rmse_train, r2_train, mse_test, rmse_test, r2_test

    def predict(self, x_new):
        """
        Faz previsões com base em novos dados de entrada.
        
        Parâmetros:
        - x_new: array-like
            Novos dados de entrada para fazer previsões.
        
        Retorna:
        - array-like
            Previsões da rede neural.
        """
        Z1 = np.dot(self.param['W1'], x_new) + self.param['b1']
        A1 = self.sigmoid(Z1)
        Z2 = np.dot(self.param['W2'], A1) + self.param['b2']
        A2 = self.sigmoid(Z2)
        return A2

### Implementação do Método 1

In [ ]:
# === CONFIGURAÇÕES ===
MAX_NEURONS = 500
PATIENCE_CONSTRUCTIVE = 250
DELTA_PERTURBATION = 0.1
VERBOSE_LEVEL = 1  # 0=silencioso, 1=resumido, 2=detalhado

# === INICIALIZAÇÃO ===
# Métricas de desempenho
mse_train_1 = []
rmse_train_1 = []
r2_train_1 = []
mse_test_1 = []
rmse_test_1 = []
r2_test_1 = []
conformity_rates_1 = []
iteration_times_1 = []

# Melhor modelo
best_metrics_1 = {
    'Train MSE': float('inf'),
    'Train RMSE': float('inf'),
    'Train R2': -float('inf'),
    'Test MSE': float('inf'),
    'Test RMSE': float('inf'),
    'Test R2': -float('inf'),
    'Best Neurons': 0,
    'Conformity Rate': 0
}
best_model_1 = None

# Early stopping construtivista
no_improve_count = 0
best_r2_so_far = -float('inf')

# Criar timestamp para esta execução
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Criar estrutura de diretórios
run_dir = Path("..") / "results" / dataset_name / "wilcar" / f"run_{timestamp}"
metrics_dir = run_dir / "metrics"
models_dir = run_dir / "models"
figures_dir = run_dir / "figures"
logs_dir = run_dir / "logs"

# Criar todos os diretórios
for directory in [metrics_dir, models_dir, figures_dir, logs_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("\n" + "=" * 70)
print(f"Diretório da execução: {run_dir.relative_to('..')}")
print("=" * 70)

# Config da EXECUÇÃO (não do dataset)
execution_config = {
    'execution_info': {
        'timestamp': timestamp,
        'dataset': dataset_name,
        'method': 'WILCAR'
    },
    'hyperparameters': {
        'max_neurons': MAX_NEURONS,
        'patience_constructive': PATIENCE_CONSTRUCTIVE,
        'patience_early_stopping': 250,
        'learning_rate': 0.1,
        'delta_perturbation': DELTA_PERTURBATION,
        'iterations': 2501
    },
    'environment': {
        'seed': SEED,
    }
}

# Salvar config da execução
execution_config_path = logs_dir / "execution_config.json"
with open(execution_config_path, 'w') as f:
    json.dump(execution_config, f, indent=2)

print(f"✅ Execution config salvo: {execution_config_path.relative_to('..')}\n")

# === LOOP DE TREINAMENTO ===
print("=" * 70)
print("TREINAMENTO - MÉTODO 1 (WILCAR)")
print("=" * 70)

ini = time.time()

for n in range(1, MAX_NEURONS + 1):
    iter_start = time.time()
    
    # Criar e treinar modelo
    if n == 1:
        model = Method1(train_inputs.T, train_targets.T, n, initial_weights=result)
    else:
        model = Method1(train_inputs.T, train_targets.T, n, previous_model=previous_model)
    
    mse_train, rmse_train, r2_train, mse_test, rmse_test, r2_test = model.train_test()
    
    # Armazenar métricas
    mse_train_1.append(mse_train)
    rmse_train_1.append(rmse_train)
    r2_train_1.append(r2_train)
    mse_test_1.append(mse_test)
    rmse_test_1.append(rmse_test)
    r2_test_1.append(r2_test)
    
    # === CÁLCULO DE GANHOS (OTIMIZADO) ===
    x_base = test_inputs.T
    y_base = model.predict(x_base)  # ✅ Calcular UMA VEZ
    
    ganhos_calculados = []
    for i in range(n_inputs):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += DELTA_PERTURBATION
        y_perturbed = model.predict(x_perturbed)
        ganho_i = (y_perturbed - y_base) / DELTA_PERTURBATION
        ganhos_calculados.append(np.mean(ganho_i))
    
    # === TAXA DE CONFORMIDADE ===
    n_conforme = 0
    n_avaliados = 0
    conformity_details = []
    
    for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados, expected_signals)):
        sinal_calculado = np.sign(ganho_calculado)
        
        if sinal_esperado != 0:  # Só avalia sinais conhecidos
            n_avaliados += 1
            conforme = (sinal_calculado == sinal_esperado)
            n_conforme += conforme
            conformity_details.append((i+1, sinal_calculado, sinal_esperado, conforme))
    
    tcs = n_conforme / n_avaliados if n_avaliados > 0 else 0
    conformity_rates_1.append(tcs)
    
    # === ATUALIZAR MELHOR MODELO ===
    if r2_test > best_metrics_1['Test R2']:
        best_metrics_1.update({
            'Train MSE': mse_train,
            'Train RMSE': rmse_train,
            'Train R2': r2_train,
            'Test MSE': mse_test,
            'Test RMSE': rmse_test,
            'Test R2': r2_test,
            'Best Neurons': n,
            'Conformity Rate': tcs
        })
        best_model_1 = model
    
    # === TEMPO DE ITERAÇÃO ===
    iter_time = time.time() - iter_start
    iteration_times_1.append(iter_time)
    
    # === OUTPUT (CONTROLADO) ===
    if VERBOSE_LEVEL >= 1:
        print(f"n={n:2d} | Train R²={r2_train:.4f} | Test R²={r2_test:.4f} | "
              f"TCS={tcs:5.1%} | Time={iter_time:5.1f}s")
    
    if VERBOSE_LEVEL >= 2:
        for var_idx, sig_calc, sig_exp, conforme in conformity_details:
            status = "✓" if conforme else "✗"
            print(f"     Var {var_idx}: {sig_calc:+.0f} (esperado {sig_exp:+.0f}) {status}")
    
    # === EARLY STOPPING CONSTRUTIVISTA ===
    if r2_test > best_r2_so_far:
        best_r2_so_far = r2_test
        no_improve_count = 0
    else:
        no_improve_count += 1
    
    if no_improve_count >= PATIENCE_CONSTRUCTIVE:
        print(f"\n⏹️ Early stopping: {PATIENCE_CONSTRUCTIVE} iterações sem melhora")
        print(f"   Melhor: n={best_metrics_1['Best Neurons']} | R²={best_r2_so_far:.4f} | TCS={best_metrics_1['Conformity Rate']:.1%}")
        break
    
    # Salvar modelo para próxima iteração
    previous_model = model

In [ ]:
logging.basicConfig(filename='wilcar_training.log', level=logging.INFO)
logging.info(f"n={n}, r2_test={r2_test:.4f}, tcs={tcs:.2%}")

### Análise da Performance do Método 1

In [ ]:
# === RESUMO FINAL ===
end = time.time()
total_time = end - ini

print("\n" + "=" * 70)
print("RESUMO - MÉTODO 1 (WILCAR SEM RESTRIÇÕES)")
print("=" * 70)
print(f"⏱️  Tempo total: {total_time:.1f}s ({total_time/60:.1f}min)")
print(f"    Tempo médio/modelo: {np.mean(iteration_times_1):.2f}s")
print(f"    Modelos treinados: {len(r2_test_1)}/{MAX_NEURONS}")
print(f"\n🏆 Melhor Modelo:")
print(f"    Neurônios: {best_metrics_1['Best Neurons']}")
print(f"    Test R²: {best_metrics_1['Test R2']:.4f}")
print(f"    Test RMSE: {best_metrics_1['Test RMSE']:.4f}")
print(f"    Taxa Conformidade: {best_metrics_1['Conformity Rate']:.1%}")
print("=" * 70)

In [ ]:
# Configurar estilo de gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)

# === RESUMO DE PERFORMANCE ===
print("\n" + "=" * 80)
print("ANÁLISE DE PERFORMANCE - MÉTODO 1 (WILCAR SEM RESTRIÇÕES)".center(80))
print("=" * 80)

# Métricas de tempo
total_time = end - ini
avg_time_per_model = np.mean(iteration_times_1)
print(f"\n⏱️  TEMPO DE EXECUÇÃO:")
print(f"    Tempo total: {total_time:.2f}s ({total_time/60:.2f}min)")
print(f"    Tempo médio/modelo: {avg_time_per_model:.2f}s")
print(f"    Modelos treinados: {len(r2_test_1)}/{MAX_NEURONS}")

# Métricas do melhor modelo
print(f"\n🏆 MELHOR MODELO:")
print(f"    Neurônios ocultos: {best_metrics_1['Best Neurons']}")
print(f"    ─" * 40)
print(f"    TREINO:")
print(f"      MSE:  {best_metrics_1['Train MSE']:.6f}")
print(f"      RMSE: {best_metrics_1['Train RMSE']:.6f}")
print(f"      R²:   {best_metrics_1['Train R2']:.6f}")
print(f"    ─" * 40)
print(f"    TESTE:")
print(f"      MSE:  {best_metrics_1['Test MSE']:.6f}")
print(f"      RMSE: {best_metrics_1['Test RMSE']:.6f}")
print(f"      R²:   {best_metrics_1['Test R2']:.6f}")

# === ANÁLISE DE GANHOS DO MELHOR MODELO ===
print(f"\n📊 ANÁLISE DE GANHOS (Melhor Modelo - n={best_metrics_1['Best Neurons']}):")
print("=" * 80)

# Configuração
x_base = test_inputs.T
delta = DELTA_PERTURBATION

# ✅ CORRIGIDO: Usar best_model_1 e calcular y_base FORA do loop
y_base = best_model_1.predict(x_base)

ganhos_calculados = []
for i in range(n_inputs):
    x_perturbed = np.copy(x_base)
    x_perturbed[i, :] += delta
    y_perturbed = best_model_1.predict(x_perturbed)
    ganho_i = (y_perturbed - y_base) / delta
    ganhos_calculados.append(np.mean(ganho_i))

# Análise de conformidade detalhada
n_conforme = 0
n_divergente = 0
n_desconhecido = 0
conformity_details = []

print(f"\n{'Variável':<12} {'Ganho Calc.':<14} {'Ganho Esp.':<14} {'Status':<12}")
print("-" * 80)

for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados, expected_signals)):
    sinal_calculado = np.sign(ganho_calculado)
    
    if sinal_esperado == 0:
        status = "Desconhecido"
        symbol = "?"
        n_desconhecido += 1
    elif sinal_calculado == sinal_esperado:
        status = "✓ Conforme"
        symbol = "✓"
        n_conforme += 1
    else:
        status = "✗ Divergente"
        symbol = "✗"
        n_divergente += 1
    
    conformity_details.append({
        'variable': i+1,
        'calculated_gain': ganho_calculado,
        'calculated_signal': int(sinal_calculado),
        'expected_signal': int(sinal_esperado),
        'conformity': status
    })
    
    print(f"Var {i+1:<8} {sinal_calculado:>+2.0f} ({ganho_calculado:>+8.4f})  "
          f"{sinal_esperado if sinal_esperado != 0 else '?':>+2}              "
          f"{symbol} {status:<12}")

# Taxa de Conformidade de Sinais (TCS)
n_avaliados = n_conforme + n_divergente
tcs = n_conforme / n_avaliados if n_avaliados > 0 else 0

print("-" * 80)
print(f"\n📈 RESUMO DE CONFORMIDADE:")
print(f"    Conformes:     {n_conforme}/{n_avaliados} ({tcs:.1%})")
print(f"    Divergentes:   {n_divergente}/{n_avaliados} ({n_divergente/n_avaliados if n_avaliados > 0 else 0:.1%})")
if n_desconhecido > 0:
    print(f"    Desconhecidos: {n_desconhecido}")
print(f"    ─" * 40)
print(f"    TCS (Taxa de Conformidade de Sinais): {tcs:.2%}")

# Atualizar best_metrics_1 com TCS
best_metrics_1['Conformity Rate'] = tcs

print("=" * 80)

# === VISUALIZAÇÕES - DASHBOARD CONSOLIDADO ===
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Criar eixo x (número de neurônios)
neurons = np.arange(1, len(r2_test_1) + 1)
best_n = best_metrics_1['Best Neurons']

# --- LINHA 1: MSE, RMSE e R² ---
# MSE
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(neurons, mse_train_1, 'o-', label='Train', alpha=0.7, markersize=4)
ax1.plot(neurons, mse_test_1, 's-', label='Test', alpha=0.7, markersize=4)
ax1.axvline(best_n, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n})')
ax1.set_xlabel('Neurônios Ocultos')
ax1.set_ylabel('MSE')
ax1.set_title('Mean Squared Error')
ax1.legend()
ax1.grid(True, alpha=0.3)

# RMSE
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(neurons, rmse_train_1, 'o-', label='Train', alpha=0.7, markersize=4)
ax2.plot(neurons, rmse_test_1, 's-', label='Test', alpha=0.7, markersize=4)
ax2.axvline(best_n, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n})')
ax2.set_xlabel('Neurônios Ocultos')
ax2.set_ylabel('RMSE')
ax2.set_title('Root Mean Squared Error')
ax2.legend()
ax2.grid(True, alpha=0.3)

# R²
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(neurons, r2_train_1, 'o-', label='Train', alpha=0.7, markersize=4)
ax3.plot(neurons, r2_test_1, 's-', label='Test', alpha=0.7, markersize=4)
ax3.axvline(best_n, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n})')
ax3.set_xlabel('Neurônios Ocultos')
ax3.set_ylabel('R²')
ax3.set_title('Coeficiente de Determinação R²')
ax3.legend()
ax3.grid(True, alpha=0.3)

# --- LINHA 2: Conformidade, Overfitting e Tempo ---
# Taxa de Conformidade
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(neurons, conformity_rates_1, 'o-', color='green', markersize=4)
ax4.axvline(best_n, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n})')
ax4.axhline(1.0, color='gray', linestyle=':', alpha=0.5, label='100%')
ax4.set_xlabel('Neurônios Ocultos')
ax4.set_ylabel('Taxa de Conformidade')
ax4.set_title('Evolução da Conformidade de Sinais')
ax4.set_ylim([0, 1.05])
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Gap Treino-Teste (Overfitting)
ax5 = fig.add_subplot(gs[1, 1])
gap_r2 = np.array(r2_train_1) - np.array(r2_test_1)
ax5.plot(neurons, gap_r2, 'o-', color='orange', markersize=4)
ax5.axvline(best_n, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n})')
ax5.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax5.set_xlabel('Neurônios Ocultos')
ax5.set_ylabel('Gap R² (Train - Test)')
ax5.set_title('Análise de Overfitting')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Tempo por Iteração
ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(neurons, iteration_times_1, 'o-', color='purple', markersize=4)
ax6.axvline(best_n, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n})')
ax6.set_xlabel('Neurônios Ocultos')
ax6.set_ylabel('Tempo (s)')
ax6.set_title('Tempo de Treinamento por Modelo')
ax6.legend()
ax6.grid(True, alpha=0.3)

# --- LINHA 3: Análise de Ganhos ---
# Ganhos Calculados vs. Esperados
ax7 = fig.add_subplot(gs[2, :2])
var_names = [f'Var {i+1}' for i in range(n_inputs)]
x_pos = np.arange(n_inputs)

# Barras dos ganhos calculados
colors = ['green' if detail['calculated_signal'] == detail['expected_signal'] or detail['expected_signal'] == 0
          else 'red' for detail in conformity_details]
bars = ax7.bar(x_pos, [d['calculated_gain'] for d in conformity_details], 
               color=colors, alpha=0.7, label='Ganho Calculado')

# Linha dos sinais esperados
expected_for_plot = [sinal if sinal != 0 else np.nan for sinal in expected_signals]
ax7.plot(x_pos, expected_for_plot, 'ko-', markersize=8, linewidth=2, 
         label='Sinal Esperado', zorder=5)

ax7.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax7.set_xlabel('Variável')
ax7.set_ylabel('Ganho')
ax7.set_title(f'Comparação de Ganhos - Melhor Modelo (n={best_n}, TCS={tcs:.1%})')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(var_names)
ax7.legend()
ax7.grid(True, alpha=0.3, axis='y')

# Adicionar anotações
for i, detail in enumerate(conformity_details):
    if detail['expected_signal'] != 0:
        symbol = '✓' if detail['calculated_signal'] == detail['expected_signal'] else '✗'
        ax7.text(i, detail['calculated_gain'], symbol, 
                ha='center', va='bottom' if detail['calculated_gain'] > 0 else 'top',
                fontsize=12, fontweight='bold')

# Sumário no último subplot
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')

summary_text = f"""
RESUMO EXECUTIVO
MÉTODO 1 (S/ RESTRIÇÕES)

Melhor Modelo:
  • Neurônios: {best_n}
  • Test R²: {best_metrics_1['Test R2']:.4f}
  • Test RMSE: {best_metrics_1['Test RMSE']:.4f}

Conformidade:
  • TCS: {tcs:.1%}
  • Conformes: {n_conforme}/{n_avaliados}
  • Divergentes: {n_divergente}/{n_avaliados}

Performance:
  • Tempo: {total_time:.1f}s
  • Modelos: {len(r2_test_1)}
  • Overfitting: {gap_r2[best_n-1]:.4f}
"""

ax8.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', 
         facecolor='wheat', alpha=0.3))

plt.suptitle('MÉTODO 1 - WILCAR SEM RESTRIÇÕES - Dashboard Completo', 
             fontsize=16, fontweight='bold', y=0.995)

plt.tight_layout()
plt.show()

print("\n✅ Visualizações geradas!")

### Salvamento de Resultados - Método 1

In [ ]:
print("\n💾 Salvando resultados do Método 1...")

# === DIRETÓRIOS ===
timestamp_1 = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir_1 = Path("..") / "results" / dataset_name / "wilcar_unconstrained" / f"run_{timestamp_1}"
metrics_dir_1 = run_dir_1 / "metrics"
models_dir_1 = run_dir_1 / "models"
figures_dir_1 = run_dir_1 / "figures"
logs_dir_1 = run_dir_1 / "logs"

# Criar diretórios
for directory in [metrics_dir_1, models_dir_1, figures_dir_1, logs_dir_1]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"📁 Diretório da execução: {run_dir_1.relative_to('..')}")

# === 1. MÉTRICAS DETALHADAS ===
metrics_detailed_1 = pd.DataFrame({
    'n_neurons': neurons,
    'mse_train': mse_train_1,
    'rmse_train': rmse_train_1,
    'r2_train': r2_train_1,
    'mse_test': mse_test_1,
    'rmse_test': rmse_test_1,
    'r2_test': r2_test_1,
    'conformity_rate': conformity_rates_1,
    'iteration_time': iteration_times_1,
    'gap_r2': gap_r2.tolist()
})

metrics_path_1 = metrics_dir_1 / "detailed_metrics.csv"
metrics_detailed_1.to_csv(metrics_path_1, index=False)
print(f"  ✓ Métricas detalhadas: {metrics_path_1.relative_to('..')}")

# === 2. RESUMO DO MELHOR MODELO ===
best_model_summary_1 = pd.DataFrame([best_metrics_1])
best_model_summary_1['dataset'] = dataset_name
best_model_summary_1['method'] = 'WILCAR_Unconstrained'
best_model_summary_1['total_time'] = total_time
best_model_summary_1['avg_time_per_model'] = avg_time_per_model
best_model_summary_1['convergence_rate'] = 1.0  # Sempre 100% (backprop sempre converge)
best_model_summary_1['timestamp'] = timestamp_1

summary_path_1 = metrics_dir_1 / "best_model_summary.csv"
best_model_summary_1.to_csv(summary_path_1, index=False)
print(f"  ✓ Melhor modelo: {summary_path_1.relative_to('..')}")

# === 3. ANÁLISE DE GANHOS ===
if best_model_1 is not None:
    ganhos_df_1 = pd.DataFrame(conformity_details)
    ganhos_df_1['dataset'] = dataset_name
    ganhos_df_1['method'] = 'WILCAR_Unconstrained'
    ganhos_df_1['n_neurons'] = best_n
    ganhos_df_1['timestamp'] = timestamp_1
    
    gains_path_1 = metrics_dir_1 / "gains_analysis.csv"
    ganhos_df_1.to_csv(gains_path_1, index=False)
    print(f"  ✓ Análise de ganhos: {gains_path_1.relative_to('..')}")

# === 4. SALVAR MELHOR MODELO ===
if best_model_1 is not None:
    model_path_1 = models_dir_1 / f"best_model_n{best_n}.pkl"
    with open(model_path_1, 'wb') as f:
        pickle.dump(best_model_1, f)
    print(f"  ✓ Modelo salvo: {model_path_1.relative_to('..')}")
    
    # Parâmetros do modelo (JSON legível)
    model_params_1 = {
        'n_neurons': best_n,
        'architecture': {
            'input_dim': n_inputs,
            'hidden_dim': best_n,
            'output_dim': 1,
            'activation': 'sigmoid',
            'constraints': None
        },
        'weights_shapes': {
            'W1': list(best_model_1.param['W1'].shape),
            'b1': list(best_model_1.param['b1'].shape),
            'W2': list(best_model_1.param['W2'].shape),
            'b2': list(best_model_1.param['b2'].shape)
        },
        'performance': {
            'train_r2': float(best_metrics_1['Train R2']),
            'test_r2': float(best_metrics_1['Test R2']),
            'test_rmse': float(best_metrics_1['Test RMSE']),
            'conformity_rate': float(best_metrics_1['Conformity Rate'])
        },
        'training': {
            'method': 'Backpropagation',
            'learning_rate': 0.1,
            'iterations': 2501,
            'patience_early_stopping': 250,
            'patience_constructive': PATIENCE_CONSTRUCTIVE
        }
    }
    
    model_params_path_1 = models_dir_1 / "model_params.json"
    with open(model_params_path_1, 'w') as f:
        json.dump(model_params_1, f, indent=2)
    print(f"  ✓ Parâmetros do modelo: {model_params_path_1.relative_to('..')}")

# === 5. SALVAR FIGURAS ===
# Dashboard principal
dashboard_path_1 = figures_dir_1 / "performance_dashboard.png"
fig.savefig(dashboard_path_1, dpi=300, bbox_inches='tight')
print(f"  ✓ Dashboard: {dashboard_path_1.relative_to('..')}")

# Alta resolução para artigo
dashboard_hires_path_1 = figures_dir_1 / "performance_dashboard_hires.pdf"
fig.savefig(dashboard_hires_path_1, dpi=600, bbox_inches='tight', format='pdf')
print(f"  ✓ Dashboard HD (PDF): {dashboard_hires_path_1.relative_to('..')}")

# Gráfico específico de ganhos (para artigo)
if best_model_1 is not None:
    fig_gains_1, ax = plt.subplots(figsize=(10, 6))
    var_names = [f'Var {i+1}' for i in range(n_inputs)]
    x_pos = np.arange(n_inputs)
    
    colors = ['green' if detail['calculated_signal'] == detail['expected_signal'] or detail['expected_signal'] == 0
              else 'red' for detail in conformity_details]
    bars = ax.bar(x_pos, [d['calculated_gain'] for d in conformity_details], 
                  color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    expected_for_plot = [sinal if sinal != 0 else np.nan for sinal in expected_signals]
    ax.plot(x_pos, expected_for_plot, 'ko-', markersize=10, linewidth=2.5, 
            label='Expected Signal', zorder=5)
    
    ax.axhline(0, color='black', linestyle='-', linewidth=0.8)
    ax.set_xlabel('Input Variable', fontsize=12, fontweight='bold')
    ax.set_ylabel('Gain (∂ŷ/∂x)', fontsize=12, fontweight='bold')
    ax.set_title(f'Gain Conformity - WILCAR Unconstrained (n={best_n}, TCS={tcs:.1%})', 
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(var_names)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    gains_fig_path_1 = figures_dir_1 / "gains_comparison.png"
    fig_gains_1.savefig(gains_fig_path_1, dpi=300, bbox_inches='tight')
    print(f"  ✓ Gráfico de ganhos: {gains_fig_path_1.relative_to('..')}")
    plt.close(fig_gains_1)

# === 6. CONFIG DA EXECUÇÃO ===
execution_config_1 = {
    'execution_info': {
        'timestamp': timestamp_1,
        'dataset': dataset_name,
        'method': 'WILCAR_Unconstrained'
    },
    'hyperparameters': {
        'max_neurons': MAX_NEURONS,
        'patience_constructive': PATIENCE_CONSTRUCTIVE,
        'patience_early_stopping': 250,
        'delta_perturbation': DELTA_PERTURBATION,
        'learning_rate': 0.1,
        'iterations': 2501
    },
    'environment': {
        'seed': SEED,
    },
    'expected_signals': expected_signals
}

execution_config_path_1 = logs_dir_1 / "execution_config.json"
with open(execution_config_path_1, 'w') as f:
    json.dump(execution_config_1, f, indent=2)
print(f"  ✓ Execution config: {execution_config_path_1.relative_to('..')}")

# === 7. LOG DE TREINAMENTO ===
log_path_1 = logs_dir_1 / "training.log"
with open(log_path_1, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("WILCAR UNCONSTRAINED TRAINING LOG\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Timestamp: {timestamp_1}\n")
    f.write(f"Dataset: {dataset_name}\n")
    f.write(f"Total time: {total_time:.2f}s ({total_time/60:.2f}min)\n")
    f.write(f"Models trained: {len(r2_test_1)}/{MAX_NEURONS}\n")
    f.write(f"Convergence rate: 100.0%\n\n")
    
    f.write("Best Model:\n")
    f.write(f"  Neurons: {best_n}\n")
    f.write(f"  Test R²: {best_metrics_1['Test R2']:.6f}\n")
    f.write(f"  Test RMSE: {best_metrics_1['Test RMSE']:.6f}\n")
    f.write(f"  Conformity Rate: {best_metrics_1['Conformity Rate']:.2%}\n\n")
    
    f.write("Iteration Details:\n")
    f.write("-" * 80 + "\n")
    for i, n in enumerate(neurons, 1):
        f.write(f"n={n:2d} | Train R²={r2_train_1[i-1]:.4f} | Test R²={r2_test_1[i-1]:.4f} | "
                f"TCS={conformity_rates_1[i-1]:.2%} | Time={iteration_times_1[i-1]:.2f}s | SUCCESS\n")

print(f"  ✓ Log de treinamento: {log_path_1.relative_to('..')}")

print("\n✅ Todos os resultados do Método 1 salvos com sucesso!")
print(f"📁 Diretório do experimento: {run_dir_1.relative_to('..')}")
print("=" * 80)

## Método 2 - WILCAR Com Restrições nos Sinais dos Ganhos - MSE

### Inicialização dos Pesos
Obtenção dos pesos iniciais entre os neurônios de entrada e o primeiro neurônio oculto, já considerando restrições nos sinais dos ganhos.

In [ ]:
# Regressão linear baseline (mesmo do Método 1)
inputs_all_cte = sm.add_constant(inputs_all, prepend=True)
res = sm.OLS(targets_all, inputs_all_cte).fit()
print(res.summary())

# Constantes da linearização
x_M = np.array([np.mean(inputs_all[:, i]) for i in range(n_inputs)])
k = np.array([res.params[i + 1] for i in range(n_inputs)])
b = res.params[0]

print("\n" + "=" * 70)
print("OTIMIZAÇÃO DE PESOS INICIAIS COM RESTRIÇÕES")
print("=" * 70)

In [ ]:
# === FUNÇÕES AUXILIARES ===

def sigmoid_safe(x):
    """Sigmoid numericamente estável."""
    return np.where(x >= 0, 
                    1 / (1 + np.exp(-np.clip(x, -500, 500))),
                    np.exp(np.clip(x, -500, 500)) / (1 + np.exp(np.clip(x, -500, 500))))

def objective_function(x, x_M, k, b, n_inputs):
    """
    Função objetivo: minimizar discrepância entre modelo linearizado e regressão linear.
    """
    w, beta = x[:-1], x[-1]
    
    # Calcular uma vez
    z = np.dot(w, x_M) + beta
    sig = sigmoid_safe(z)
    sig_prime = sig * (1 - sig)
    
    # Equação do intercepto
    eq_intercept = sig - sig_prime * np.dot(w, x_M) - b
    
    # Equações dos coeficientes
    eq_coefs = sig_prime * w - k
    
    # Soma dos quadrados
    return eq_intercept**2 + np.sum(eq_coefs**2)

def gain_constraint_init(x, idx, x_base, expected_signal, delta=0.1):
    """
    Constraint para ganho na inicialização.
    
    Args:
        x: vetor [w1, w2, ..., wn, beta]
        idx: índice da variável
        x_base: ponto base para calcular ganho
        expected_signal: sinal esperado (+1, -1, ou 0)
        delta: perturbação
    
    Returns:
        valor positivo se constraint satisfeita
    """
    if expected_signal == 0:
        return 1.0  # Sempre satisfeito
    
    w, beta = x[:-1], x[-1]
    
    # Calcular ganho
    x_pert = x_base.copy()
    x_pert[idx] += delta
    
    y_base = sigmoid_safe(np.dot(w, x_base) + beta)
    y_pert = sigmoid_safe(np.dot(w, x_pert) + beta)
    
    gain = (y_pert - y_base) / delta
    
    # Retornar constraint
    margin = 0.05  # Margin mais rigoroso que 0.01
    return expected_signal * gain - margin

In [ ]:
# === PONTO BASE PARA CÁLCULO DE GANHOS ===
# Usar média dos dados de treino (mais representativo)
x_base_init = np.mean(train_inputs, axis=0)

print(f"Ponto base para ganhos: média dos dados de treino")
print(f"   Valores: {x_base_init}")

In [ ]:
# === CONSTRAINTS ===
constraints_init = []
for i in range(n_inputs):
    if expected_signals[i] != 0:  # Só restringe sinais conhecidos
        # ✅ CORREÇÃO Bug #3: Usar closure correta com valores default
        constraints_init.append({
            'type': 'ineq',
            'fun': lambda x, idx=i, sig=expected_signals[i]: 
                   gain_constraint_init(x, idx, x_base_init, sig)
        })

print(f"\n✅ Constraints criadas: {len(constraints_init)} restrições de ganho")
for i in range(n_inputs):
    if expected_signals[i] != 0:
        print(f"   Var {i+1}: ganho deve ser {'positivo' if expected_signals[i] > 0 else 'negativo'}")

In [ ]:
# === OTIMIZAÇÃO COM MÚLTIPLAS TENTATIVAS ===
best_result_init = None
best_residual_init = float('inf')

print(f"\n🔄 Tentando otimização com restrições (até 10 tentativas)...")

for trial in range(10):
    # Chute inicial
    if trial == 0:
        # Tentativa 1: usar coeficientes da regressão
        x0 = np.append(k, b)
    elif trial == 1:
        # Tentativa 2: pequena perturbação nos coeficientes
        x0 = np.append(k * (1 + 0.1 * np.random.randn(n_inputs)), b * (1 + 0.1 * np.random.randn()))
    else:
        # Outras tentativas: aleatório
        x0 = np.random.randn(n_inputs + 1) * 0.5
    
    # Otimização
    result_trial = minimize(
        objective_function,
        x0,
        args=(x_M, k, b, n_inputs),
        method='SLSQP',
        constraints=constraints_init,
        options={'maxiter': 5000, 'ftol': 1e-9},
        tol=1e-9
    )
    
    if result_trial.success:
        # Calcular resíduo
        residual = objective_function(result_trial.x, x_M, k, b, n_inputs)
        
        # Verificar se ganhos foram satisfeitos
        ganhos_ok = True
        for i in range(n_inputs):
            if expected_signals[i] != 0:
                constraint_val = gain_constraint_init(result_trial.x, i, x_base_init, expected_signals[i])
                if constraint_val < -1e-6:  # Pequena tolerância numérica
                    ganhos_ok = False
                    break
        
        if ganhos_ok and residual < best_residual_init:
            best_result_init = result_trial
            best_residual_init = residual
            print(f"   ✓ Tentativa {trial+1}: sucesso (resíduo={residual:.2e})")
        else:
            if not ganhos_ok:
                print(f"   ✗ Tentativa {trial+1}: ganhos não satisfeitos")
            else:
                print(f"   - Tentativa {trial+1}: resíduo pior ({residual:.2e})")
    else:
        print(f"   ✗ Tentativa {trial+1}: falhou ({result_trial.message})")
    
    # Se encontrou solução muito boa, pode parar
    if best_residual_init < 1e-8:
        break

In [ ]:
# === RESULTADO FINAL ===
if best_result_init is not None:
    print(f"\n✅ Otimização concluída com sucesso!")
    print(f"   Melhor resíduo: {best_residual_init:.2e}")
    print(f"   Pesos iniciais: {best_result_init.x}")
    result_constrained = best_result_init.x
else:
    print(f"\n⚠️ Otimização não convergiu com restrições!")
    print(f"   Usando fallback: pesos da regressão linear")
    result_constrained = np.append(k, b)

print("=" * 70)

### Avaliação dos Sinais dos Ganhos do Modelo Inicial
O modelo inicial corresponde a um Perceptron.

In [ ]:
### Avaliação dos Sinais dos Ganhos do Modelo Inicial

print("\n" + "=" * 70)
print("AVALIAÇÃO DE GANHOS - MODELO INICIAL (PERCEPTRON COM RESTRIÇÕES)")
print("=" * 70)

# Usar média do conjunto de teste como ponto de avaliação
x_base_eval = np.mean(test_inputs, axis=0).reshape(-1, 1)

def calculate_and_compare_gains_simple(weights, x_base, expected_signals, n_inputs, delta=0.1):
    """
    Calcula ganhos de um perceptron simples.
    """
    w, beta = weights[:-1], weights[-1]
    
    # Garantir que x_base é 1D
    if x_base.ndim > 1:
        x_base = x_base.flatten()
    
    y_base = sigmoid_safe(np.dot(w, x_base) + beta)
    
    print(f"\n{'Variável':<12} {'Ganho Calc.':<14} {'Ganho Esp.':<14} {'Status':<12}")
    print("-" * 70)
    
    n_conforme = 0
    n_avaliados = 0
    
    for i in range(n_inputs):
        x_var = x_base.copy()
        x_var[i] += delta
        y_new = sigmoid_safe(np.dot(w, x_var) + beta)
        gain = (y_new - y_base) / delta
        signal_calc = np.sign(gain)
        signal_exp = expected_signals[i]
        
        if signal_exp != 0:
            n_avaliados += 1
            if signal_calc == signal_exp:
                status = "✓ Conforme"
                n_conforme += 1
            else:
                status = "✗ Divergente"
        else:
            status = "? Desconhecido"
        
        print(f"Var {i+1:<8} {signal_calc:>+2.0f} ({gain:>+8.4f})  "
              f"{signal_exp if signal_exp != 0 else '?':>+2}              "
              f"{status:<12}")
    
    print("-" * 70)
    if n_avaliados > 0:
        tcs = n_conforme / n_avaliados
        print(f"Taxa de Conformidade: {n_conforme}/{n_avaliados} ({tcs:.1%})")
    print("=" * 70)

calculate_and_compare_gains_simple(result_constrained, x_base_eval, expected_signals, n_inputs)

### Definição / Implementação do Método 2

In [ ]:
### Definição da Classe do Método 2 (Com Restrições)

class Method2:
    """
    WILCAR com restrições nos sinais dos ganhos usando scipy.optimize.minimize.
    """
    
    def __init__(self, x, y, n, expected_signals, previous_model=None, initial_weights=None):
        self.x = x  # Shape: (n_features, n_samples)
        self.y = y.reshape(1, -1)  # Shape: (1, n_samples)
        self.n = n
        self.expected_signals = expected_signals
        self.n_inputs = x.shape[0]
        
        # Inicializar parâmetros
        if previous_model is None:
            self.param = self.initialize_parameters(n, initial_weights)
        else:
            self.param = self.expand_network(previous_model, n)
        
        self.dims = [x.shape[0], n, 1]
        self.sam = y.shape[0]
    
    def initialize_parameters(self, n, initial_weights):
        """Inicializa parâmetros (primeiro neurônio usa pesos com restrições)."""
        param = {
            'W1': np.random.randn(n, self.n_inputs) * np.sqrt(1. / self.n_inputs),
            'b1': np.zeros((n, 1)),
            'W2': np.random.randn(1, n) * np.sqrt(1. / n),
            'b2': np.zeros((1, 1))
        }
        
        if initial_weights is not None:
            param['W1'][0, :] = initial_weights[:-1]
            param['b1'][0, :] = initial_weights[-1]
        
        return param
    
    def expand_network(self, previous_model, n):
        """Expande rede adicionando neurônios."""
        new_param = copy.deepcopy(previous_model.param)
        
        if n > previous_model.n:
            # Adicionar novo neurônio
            additional_weights = np.random.randn(1, self.n_inputs) * np.sqrt(1. / self.n_inputs)
            new_param['W1'] = np.vstack([new_param['W1'], additional_weights])
            new_param['b1'] = np.vstack([new_param['b1'], np.zeros((1, 1))])
            new_param['W2'] = np.hstack([new_param['W2'], np.random.randn(1, 1) * np.sqrt(1. / n)])
        
        return new_param
    
    def sigmoid(self, Z):
        """Sigmoid numericamente estável."""
        return np.where(Z >= 0,
                       1 / (1 + np.exp(-np.clip(Z, -500, 500))),
                       np.exp(np.clip(Z, -500, 500)) / (1 + np.exp(np.clip(Z, -500, 500))))
    
    def forward(self):
        """Forward pass com cache de ativações."""
        self.Z1 = self.param['W1'].dot(self.x) + self.param['b1']
        self.A1 = self.sigmoid(self.Z1)
        self.Z2 = self.param['W2'].dot(self.A1) + self.param['b2']
        self.A2 = self.sigmoid(self.Z2)
        return self.A2
    
    def predict(self, x_new):
        """Predição em novos dados."""
        Z1 = np.dot(self.param['W1'], x_new) + self.param['b1']
        A1 = self.sigmoid(Z1)
        Z2 = np.dot(self.param['W2'], A1) + self.param['b2']
        A2 = self.sigmoid(Z2)
        return A2
    
    def calculate_gains(self, x_data, delta=0.1):
        """
        Calcula ganhos para todas as variáveis.
        
        Args:
            x_data: dados de entrada (n_features, n_samples)
            delta: perturbação
        
        Returns:
            array de ganhos (n_inputs,)
        """
        y_base = self.predict(x_data)
        gains = []
        
        for i in range(self.n_inputs):
            x_pert = np.copy(x_data)
            x_pert[i, :] += delta
            y_pert = self.predict(x_pert)
            gain_i = np.mean((y_pert - y_base) / delta)
            gains.append(gain_i)
        
        return np.array(gains)
    
    def params_to_vector(self):
        """Converte parâmetros dict → vector."""
        return np.concatenate([
            self.param['W1'].flatten(),
            self.param['b1'].flatten(),
            self.param['W2'].flatten(),
            self.param['b2'].flatten()
        ])
    
    def vector_to_params(self, vector):
        """Converte vector → parâmetros dict."""
        n_W1 = self.n * self.n_inputs
        n_b1 = self.n
        n_W2 = self.n
        n_b2 = 1
        
        W1 = vector[:n_W1].reshape(self.n, self.n_inputs)
        b1 = vector[n_W1:n_W1+n_b1].reshape(self.n, 1)
        W2 = vector[n_W1+n_b1:n_W1+n_b1+n_W2].reshape(1, self.n)
        b2 = vector[-n_b2:].reshape(1, 1)
        
        return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
    
    def objective_mse(self, param_vector):
        """Função objetivo: MSE."""
        self.param = self.vector_to_params(param_vector)
        y_pred = self.forward()
        return np.mean((y_pred - self.y)**2)
    
    def constraint_gain(self, param_vector, var_idx, expected_sign, x_constraint, delta=0.1):
        """
        Constraint para ganho de uma variável.
        
        Returns:
            valor >= 0 se constraint satisfeita
        """
        if expected_sign == 0:
            return 1.0
        
        # Temporariamente atualizar parâmetros
        param_temp = self.vector_to_params(param_vector)
        
        # Calcular ganho
        Z1 = np.dot(param_temp['W1'], x_constraint) + param_temp['b1']
        A1 = self.sigmoid(Z1)
        Z2 = np.dot(param_temp['W2'], A1) + param_temp['b2']
        y_base = self.sigmoid(Z2)
        
        x_pert = x_constraint.copy()
        x_pert[var_idx, :] += delta
        
        Z1_p = np.dot(param_temp['W1'], x_pert) + param_temp['b1']
        A1_p = self.sigmoid(Z1_p)
        Z2_p = np.dot(param_temp['W2'], A1_p) + param_temp['b2']
        y_pert = self.sigmoid(Z2_p)
        
        gain = np.mean((y_pert - y_base) / delta)
        
        # Constraint: expected_sign * gain >= margin
        margin = 0.05
        return expected_sign * gain - margin
    
    def train_with_constraints(self, max_iter=1000, tol=1e-6, verbose=False):
        """
        Treina a rede usando scipy.optimize.minimize com constraints.
        
        Returns:
            success (bool), n_iterations (int)
        """
        # Construir constraints
        constraints = []
        for i in range(self.n_inputs):
            if self.expected_signals[i] != 0:
                # ✅ Closure correta com defaults
                constraints.append({
                    'type': 'ineq',
                    'fun': lambda pv, idx=i, sig=self.expected_signals[i]: 
                           self.constraint_gain(pv, idx, sig, self.x)
                })
        
        # Vetor inicial
        x0 = self.params_to_vector()
        
        # Otimização
        result = minimize(
            self.objective_mse,
            x0,
            method='SLSQP',
            constraints=constraints,
            options={'maxiter': max_iter, 'disp': verbose, 'ftol': tol},
            tol=tol
        )
        
        if result.success:
            # Atualizar parâmetros
            self.param = self.vector_to_params(result.x)
        
        return result.success, result.nit
    
    def evaluate(self, test_inputs, test_targets):
        """Avalia modelo em dados de teste."""
        y_pred = self.predict(test_inputs.T)
        
        mse_val = np.mean((test_targets - y_pred.flatten())**2)
        rmse_val = np.sqrt(mse_val)
        r2_val = r2_score(test_targets, y_pred.flatten())
        
        return mse_val, rmse_val, r2_val

print("✅ Classe Method2 definida")

In [ ]:
### Implementação do Método 2 - OTIMIZADO COM RESTRIÇÕES

# === CONFIGURAÇÕES ===
MAX_NEURONS_2 = 100  # Pode ser menor inicialmente para testar
PATIENCE_CONSTRUCTIVE_2 = 50  # Mais paciente (otimização é mais lenta)
DELTA_PERTURBATION_2 = 0.1
VERBOSE_LEVEL_2 = 1

# === INICIALIZAÇÃO ===
mse_train_2 = []
rmse_train_2 = []
r2_train_2 = []
mse_test_2 = []
rmse_test_2 = []
r2_test_2 = []
conformity_rates_2 = []
iteration_times_2 = []
optimization_success_2 = []  # Tracking se minimize convergiu

best_metrics_2 = {
    'Train MSE': float('inf'),
    'Train RMSE': float('inf'),
    'Train R2': -float('inf'),
    'Test MSE': float('inf'),
    'Test RMSE': float('inf'),
    'Test R2': -float('inf'),
    'Best Neurons': 0,
    'Conformity Rate': 0
}
best_model_2 = None

no_improve_count_2 = 0
best_r2_so_far_2 = -float('inf')

# === LOOP DE TREINAMENTO ===
print("\n" + "=" * 70)
print("TREINAMENTO - MÉTODO 2 (WILCAR COM RESTRIÇÕES - MSE)")
print("=" * 70)

ini_2 = time.time()

for n in range(1, MAX_NEURONS_2 + 1):
    iter_start = time.time()
    
    # Criar modelo
    if n == 1:
        model_2 = Method2(train_inputs.T, train_targets.T, n, 
                         expected_signals, initial_weights=result_constrained)
    else:
        model_2 = Method2(train_inputs.T, train_targets.T, n,
                         expected_signals, previous_model=previous_model_2)
    
    # Treinar com restrições
    success, n_iter = model_2.train_with_constraints(
        max_iter=1000, 
        tol=1e-6,
        verbose=(VERBOSE_LEVEL_2 >= 2)
    )
    
    optimization_success_2.append(success)
    
    if not success:
        print(f"n={n:2d} | ⚠️ Otimização não convergiu - pulando")
        # Adicionar valores inválidos para manter consistência de índices
        mse_train_2.append(np.nan)
        rmse_train_2.append(np.nan)
        r2_train_2.append(np.nan)
        mse_test_2.append(np.nan)
        rmse_test_2.append(np.nan)
        r2_test_2.append(np.nan)
        conformity_rates_2.append(np.nan)
        iteration_times_2.append(time.time() - iter_start)
        continue
    
    # Avaliar treino
    y_train_pred = model_2.forward()
    mse_train = np.mean((y_train_pred - model_2.y)**2)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(model_2.y.flatten(), y_train_pred.flatten())
    
    # Avaliar teste
    mse_test, rmse_test, r2_test = model_2.evaluate(test_inputs, test_targets)
    
    # Armazenar métricas
    mse_train_2.append(mse_train)
    rmse_train_2.append(rmse_train)
    r2_train_2.append(r2_train)
    mse_test_2.append(mse_test)
    rmse_test_2.append(rmse_test)
    r2_test_2.append(r2_test)
    
    # === CÁLCULO DE GANHOS (TESTE) ===
    gains_test = model_2.calculate_gains(test_inputs.T, delta=DELTA_PERTURBATION_2)
    
    # Taxa de conformidade
    n_conforme = 0
    n_avaliados = 0
    
    for i in range(n_inputs):
        if expected_signals[i] != 0:
            n_avaliados += 1
            if np.sign(gains_test[i]) == expected_signals[i]:
                n_conforme += 1
    
    tcs = n_conforme / n_avaliados if n_avaliados > 0 else 0
    conformity_rates_2.append(tcs)
    
    # === ATUALIZAR MELHOR MODELO ===
    if r2_test > best_metrics_2['Test R2']:
        best_metrics_2.update({
            'Train MSE': mse_train,
            'Train RMSE': rmse_train,
            'Train R2': r2_train,
            'Test MSE': mse_test,
            'Test RMSE': rmse_test,
            'Test R2': r2_test,
            'Best Neurons': n,
            'Conformity Rate': tcs
        })
        best_model_2 = model_2
    
    # Tempo
    iter_time = time.time() - iter_start
    iteration_times_2.append(iter_time)
    
    # Output
    if VERBOSE_LEVEL_2 >= 1:
        print(f"n={n:2d} | Train R²={r2_train:.4f} | Test R²={r2_test:.4f} | "
              f"TCS={tcs:5.1%} | Time={iter_time:5.1f}s | Iters={n_iter}")
    
    # Early stopping construtivista
    if r2_test > best_r2_so_far_2:
        best_r2_so_far_2 = r2_test
        no_improve_count_2 = 0
    else:
        no_improve_count_2 += 1
    
    if no_improve_count_2 >= PATIENCE_CONSTRUCTIVE_2:
        print(f"\n⏹️ Early stopping: {PATIENCE_CONSTRUCTIVE_2} iterações sem melhora")
        print(f"   Melhor: n={best_metrics_2['Best Neurons']} | R²={best_r2_so_far_2:.4f} | TCS={best_metrics_2['Conformity Rate']:.1%}")
        break
    
    # Salvar para próxima iteração
    previous_model_2 = model_2

end_2 = time.time()
total_time_2 = end_2 - ini_2

### Análise da Performance do Método 2

In [ ]:
# === RESUMO FINAL ===
print("\n" + "=" * 70)
print("RESUMO - MÉTODO 2 (WILCAR COM RESTRIÇÕES - MSE)")
print("=" * 70)
print(f"⏱️  Tempo total: {total_time_2:.1f}s ({total_time_2/60:.1f}min)")
print(f"    Tempo médio/modelo: {np.nanmean(iteration_times_2):.2f}s")
print(f"    Modelos treinados: {len([x for x in optimization_success_2 if x])}/{MAX_NEURONS_2}")
print(f"\n🏆 Melhor Modelo:")
print(f"    Neurônios: {best_metrics_2['Best Neurons']}")
print(f"    Test R²: {best_metrics_2['Test R2']:.4f}")
print(f"    Test RMSE: {best_metrics_2['Test RMSE']:.4f}")
print(f"    Taxa Conformidade: {best_metrics_2['Conformity Rate']:.1%}")
print("=" * 70)

In [ ]:
# Configurar estilo
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)

print("\n" + "=" * 80)
print("ANÁLISE DE PERFORMANCE - MÉTODO 2 (WILCAR COM RESTRIÇÕES - MSE)".center(80))
print("=" * 80)

# Métricas de tempo
total_time_2 = end_2 - ini_2
avg_time_per_model_2 = np.nanmean(iteration_times_2)
n_models_trained_2 = len([x for x in optimization_success_2 if x])

print(f"\n⏱️  TEMPO DE EXECUÇÃO:")
print(f"    Tempo total: {total_time_2:.2f}s ({total_time_2/60:.2f}min)")
print(f"    Tempo médio/modelo: {avg_time_per_model_2:.2f}s")
print(f"    Modelos treinados: {n_models_trained_2}/{MAX_NEURONS_2}")
print(f"    Taxa de convergência: {n_models_trained_2/len(optimization_success_2):.1%}")

# Métricas do melhor modelo
print(f"\n🏆 MELHOR MODELO:")
print(f"    Neurônios ocultos: {best_metrics_2['Best Neurons']}")
print(f"    ─" * 40)
print(f"    TREINO:")
print(f"      MSE:  {best_metrics_2['Train MSE']:.6f}")
print(f"      RMSE: {best_metrics_2['Train RMSE']:.6f}")
print(f"      R²:   {best_metrics_2['Train R2']:.6f}")
print(f"    ─" * 40)
print(f"    TESTE:")
print(f"      MSE:  {best_metrics_2['Test MSE']:.6f}")
print(f"      RMSE: {best_metrics_2['Test RMSE']:.6f}")
print(f"      R²:   {best_metrics_2['Test R2']:.6f}")

# === ANÁLISE DE GANHOS DO MELHOR MODELO ===
print(f"\n📊 ANÁLISE DE GANHOS (Melhor Modelo - n={best_metrics_2['Best Neurons']}):")
print("=" * 80)

if best_model_2 is not None:
    # Calcular ganhos no teste
    y_base_2 = best_model_2.predict(test_inputs.T)
    
    ganhos_calculados_2 = []
    for i in range(n_inputs):
        x_perturbed = np.copy(test_inputs.T)
        x_perturbed[i, :] += DELTA_PERTURBATION_2
        y_perturbed = best_model_2.predict(x_perturbed)
        ganho_i = (y_perturbed - y_base_2) / DELTA_PERTURBATION_2
        ganhos_calculados_2.append(np.mean(ganho_i))
    
    # Análise de conformidade detalhada
    n_conforme = 0
    n_divergente = 0
    n_desconhecido = 0
    conformity_details_2 = []
    
    print(f"\n{'Variável':<12} {'Ganho Calc.':<14} {'Ganho Esp.':<14} {'Status':<12}")
    print("-" * 80)
    
    for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados_2, expected_signals)):
        sinal_calculado = np.sign(ganho_calculado)
        
        if sinal_esperado == 0:
            status = "Desconhecido"
            symbol = "?"
            n_desconhecido += 1
        elif sinal_calculado == sinal_esperado:
            status = "✓ Conforme"
            symbol = "✓"
            n_conforme += 1
        else:
            status = "✗ Divergente"
            symbol = "✗"
            n_divergente += 1
        
        conformity_details_2.append({
            'variable': i+1,
            'calculated_gain': ganho_calculado,
            'calculated_signal': int(sinal_calculado),
            'expected_signal': int(sinal_esperado),
            'conformity': status
        })
        
        print(f"Var {i+1:<8} {sinal_calculado:>+2.0f} ({ganho_calculado:>+8.4f})  "
              f"{sinal_esperado if sinal_esperado != 0 else '?':>+2}              "
              f"{symbol} {status:<12}")
    
    # Taxa de Conformidade de Sinais (TCS)
    n_avaliados = n_conforme + n_divergente
    tcs_final_2 = n_conforme / n_avaliados if n_avaliados > 0 else 0
    
    print("-" * 80)
    print(f"\n📈 RESUMO DE CONFORMIDADE:")
    print(f"    Conformes:     {n_conforme}/{n_avaliados} ({tcs_final_2:.1%})")
    print(f"    Divergentes:   {n_divergente}/{n_avaliados} ({n_divergente/n_avaliados if n_avaliados > 0 else 0:.1%})")
    if n_desconhecido > 0:
        print(f"    Desconhecidos: {n_desconhecido}")
    print(f"    ─" * 40)
    print(f"    TCS (Taxa de Conformidade de Sinais): {tcs_final_2:.2%}")
    
    # Atualizar best_metrics_2 com TCS final
    best_metrics_2['Conformity Rate'] = tcs_final_2
    
    print("=" * 80)
else:
    print("⚠️ Nenhum modelo válido foi treinado!")
    tcs_final_2 = 0

# === VISUALIZAÇÕES - DASHBOARD CONSOLIDADO ===
fig2 = plt.figure(figsize=(16, 12))
gs2 = fig2.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Criar eixo x (número de neurônios) - filtrar NaNs
neurons_2 = np.arange(1, len(r2_test_2) + 1)
best_n_2 = best_metrics_2['Best Neurons']

# Filtrar valores válidos (não NaN)
valid_indices = [i for i, x in enumerate(optimization_success_2) if x]

# --- LINHA 1: MSE, RMSE e R² ---
# MSE
ax1_2 = fig2.add_subplot(gs2[0, 0])
if len(valid_indices) > 0:
    neurons_valid = [neurons_2[i] for i in valid_indices]
    mse_train_valid = [mse_train_2[i] for i in valid_indices]
    mse_test_valid = [mse_test_2[i] for i in valid_indices]
    
    ax1_2.plot(neurons_valid, mse_train_valid, 'o-', label='Train', alpha=0.7, markersize=4)
    ax1_2.plot(neurons_valid, mse_test_valid, 's-', label='Test', alpha=0.7, markersize=4)
    ax1_2.axvline(best_n_2, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n_2})')
ax1_2.set_xlabel('Neurônios Ocultos')
ax1_2.set_ylabel('MSE')
ax1_2.set_title('Mean Squared Error')
ax1_2.legend()
ax1_2.grid(True, alpha=0.3)

# RMSE
ax2_2 = fig2.add_subplot(gs2[0, 1])
if len(valid_indices) > 0:
    rmse_train_valid = [rmse_train_2[i] for i in valid_indices]
    rmse_test_valid = [rmse_test_2[i] for i in valid_indices]
    
    ax2_2.plot(neurons_valid, rmse_train_valid, 'o-', label='Train', alpha=0.7, markersize=4)
    ax2_2.plot(neurons_valid, rmse_test_valid, 's-', label='Test', alpha=0.7, markersize=4)
    ax2_2.axvline(best_n_2, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n_2})')
ax2_2.set_xlabel('Neurônios Ocultos')
ax2_2.set_ylabel('RMSE')
ax2_2.set_title('Root Mean Squared Error')
ax2_2.legend()
ax2_2.grid(True, alpha=0.3)

# R²
ax3_2 = fig2.add_subplot(gs2[0, 2])
if len(valid_indices) > 0:
    r2_train_valid = [r2_train_2[i] for i in valid_indices]
    r2_test_valid = [r2_test_2[i] for i in valid_indices]
    
    ax3_2.plot(neurons_valid, r2_train_valid, 'o-', label='Train', alpha=0.7, markersize=4)
    ax3_2.plot(neurons_valid, r2_test_valid, 's-', label='Test', alpha=0.7, markersize=4)
    ax3_2.axvline(best_n_2, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n_2})')
ax3_2.set_xlabel('Neurônios Ocultos')
ax3_2.set_ylabel('R²')
ax3_2.set_title('Coeficiente de Determinação R²')
ax3_2.legend()
ax3_2.grid(True, alpha=0.3)

# --- LINHA 2: Conformidade, Overfitting e Tempo ---
# Taxa de Conformidade
ax4_2 = fig2.add_subplot(gs2[1, 0])
if len(valid_indices) > 0:
    conformity_valid = [conformity_rates_2[i] for i in valid_indices]
    
    ax4_2.plot(neurons_valid, conformity_valid, 'o-', color='green', markersize=4)
    ax4_2.axvline(best_n_2, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n_2})')
    ax4_2.axhline(1.0, color='gray', linestyle=':', alpha=0.5, label='100%')
ax4_2.set_xlabel('Neurônios Ocultos')
ax4_2.set_ylabel('Taxa de Conformidade')
ax4_2.set_title('Evolução da Conformidade de Sinais')
ax4_2.set_ylim([0, 1.05])
ax4_2.legend()
ax4_2.grid(True, alpha=0.3)
ax4_2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# ✅ MUDANÇA: Gap Treino-Teste (Overfitting) - ao invés de Taxa de Convergência
ax5_2 = fig2.add_subplot(gs2[1, 1])
if len(valid_indices) > 0:
    gap_r2_valid = [r2_train_2[i] - r2_test_2[i] for i in valid_indices]
    
    ax5_2.plot(neurons_valid, gap_r2_valid, 'o-', color='orange', markersize=4)
    ax5_2.axvline(best_n_2, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n_2})')
    ax5_2.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax5_2.set_xlabel('Neurônios Ocultos')
ax5_2.set_ylabel('Gap R² (Train - Test)')
ax5_2.set_title('Análise de Overfitting')
ax5_2.legend()
ax5_2.grid(True, alpha=0.3)

# Tempo por Iteração
ax6_2 = fig2.add_subplot(gs2[1, 2])
if len(valid_indices) > 0:
    time_valid = [iteration_times_2[i] for i in valid_indices]
    
    ax6_2.plot(neurons_valid, time_valid, 'o-', color='purple', markersize=4)
    ax6_2.axvline(best_n_2, color='red', linestyle='--', alpha=0.5, label=f'Best (n={best_n_2})')
ax6_2.set_xlabel('Neurônios Ocultos')
ax6_2.set_ylabel('Tempo (s)')
ax6_2.set_title('Tempo de Treinamento por Modelo')
ax6_2.legend()
ax6_2.grid(True, alpha=0.3)

# --- LINHA 3: Análise de Ganhos ---
# Ganhos Calculados vs. Esperados
if best_model_2 is not None:
    ax7_2 = fig2.add_subplot(gs2[2, :2])
    var_names = [f'Var {i+1}' for i in range(n_inputs)]
    x_pos = np.arange(n_inputs)
    
    # Barras dos ganhos calculados
    colors = ['green' if detail['calculated_signal'] == detail['expected_signal'] or detail['expected_signal'] == 0
              else 'red' for detail in conformity_details_2]
    bars = ax7_2.bar(x_pos, [d['calculated_gain'] for d in conformity_details_2], 
                   color=colors, alpha=0.7, label='Ganho Calculado')
    
    # Linha dos sinais esperados
    expected_for_plot = [sinal if sinal != 0 else np.nan for sinal in expected_signals]
    ax7_2.plot(x_pos, expected_for_plot, 'ko-', markersize=8, linewidth=2, 
             label='Sinal Esperado', zorder=5)
    
    ax7_2.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax7_2.set_xlabel('Variável')
    ax7_2.set_ylabel('Ganho')
    ax7_2.set_title(f'Comparação de Ganhos - Melhor Modelo (n={best_n_2}, TCS={tcs_final_2:.1%})')
    ax7_2.set_xticks(x_pos)
    ax7_2.set_xticklabels(var_names)
    ax7_2.legend()
    ax7_2.grid(True, alpha=0.3, axis='y')
    
    # Adicionar anotações
    for i, detail in enumerate(conformity_details_2):
        if detail['expected_signal'] != 0:
            symbol = '✓' if detail['calculated_signal'] == detail['expected_signal'] else '✗'
            ax7_2.text(i, detail['calculated_gain'], symbol, 
                    ha='center', va='bottom' if detail['calculated_gain'] > 0 else 'top',
                    fontsize=12, fontweight='bold')

# Sumário no último subplot
ax8_2 = fig2.add_subplot(gs2[2, 2])
ax8_2.axis('off')

if len(valid_indices) > 0:
    gap_at_best = r2_train_2[best_n_2-1] - r2_test_2[best_n_2-1] if best_n_2 <= len(r2_train_2) else 0
else:
    gap_at_best = 0

summary_text_2 = f"""
RESUMO EXECUTIVO
MÉTODO 2 (C/ RESTRIÇÕES MSE)

Melhor Modelo:
  • Neurônios: {best_n_2}
  • Test R²: {best_metrics_2['Test R2']:.4f}
  • Test RMSE: {best_metrics_2['Test RMSE']:.4f}

Conformidade:
  • TCS: {tcs_final_2:.1%}
  • Conformes: {n_conforme}/{n_avaliados}
  • Divergentes: {n_divergente}/{n_avaliados}

Performance:
  • Tempo: {total_time_2:.1f}s
  • Modelos: {n_models_trained_2}/{MAX_NEURONS_2}
  • Convergência: {n_models_trained_2/len(optimization_success_2):.1%}
  • Overfitting: {gap_at_best:.4f}
"""

ax8_2.text(0.1, 0.5, summary_text_2, fontsize=11, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', 
         facecolor='lightblue', alpha=0.3))

plt.suptitle('MÉTODO 2 - WILCAR COM RESTRIÇÕES (MSE) - Dashboard Completo', 
             fontsize=16, fontweight='bold', y=0.995)

plt.tight_layout()
plt.show()

print("\n✅ Visualizações geradas!")

### Salvamento de Resultados - Método 2

In [ ]:
print("\n💾 Salvando resultados do Método 2...")

# === DIRETÓRIOS ===
timestamp_2 = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir_2 = Path("..") / "results" / dataset_name / "wilcar_constrained_mse" / f"run_{timestamp_2}"
metrics_dir_2 = run_dir_2 / "metrics"
models_dir_2 = run_dir_2 / "models"
figures_dir_2 = run_dir_2 / "figures"
logs_dir_2 = run_dir_2 / "logs"

# Criar diretórios
for directory in [metrics_dir_2, models_dir_2, figures_dir_2, logs_dir_2]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"📁 Diretório da execução: {run_dir_2.relative_to('..')}")

# === 1. MÉTRICAS DETALHADAS ===
metrics_detailed_2 = pd.DataFrame({
    'n_neurons': neurons_2,
    'mse_train': mse_train_2,
    'rmse_train': rmse_train_2,
    'r2_train': r2_train_2,
    'mse_test': mse_test_2,
    'rmse_test': rmse_test_2,
    'r2_test': r2_test_2,
    'conformity_rate': conformity_rates_2,
    'iteration_time': iteration_times_2,
    'optimization_success': optimization_success_2,
    'gap_r2': [r2_train_2[i] - r2_test_2[i] if not np.isnan(r2_train_2[i]) else np.nan 
               for i in range(len(r2_train_2))]
})

metrics_path_2 = metrics_dir_2 / "detailed_metrics.csv"
metrics_detailed_2.to_csv(metrics_path_2, index=False)
print(f"  ✓ Métricas detalhadas: {metrics_path_2.relative_to('..')}")

# === 2. RESUMO DO MELHOR MODELO ===
best_model_summary_2 = pd.DataFrame([best_metrics_2])
best_model_summary_2['dataset'] = dataset_name
best_model_summary_2['method'] = 'WILCAR_Constrained_MSE'
best_model_summary_2['total_time'] = total_time_2
best_model_summary_2['avg_time_per_model'] = avg_time_per_model_2
best_model_summary_2['convergence_rate'] = n_models_trained_2 / len(optimization_success_2)
best_model_summary_2['timestamp'] = timestamp_2

summary_path_2 = metrics_dir_2 / "best_model_summary.csv"
best_model_summary_2.to_csv(summary_path_2, index=False)
print(f"  ✓ Melhor modelo: {summary_path_2.relative_to('..')}")

# === 3. ANÁLISE DE GANHOS ===
if best_model_2 is not None:
    ganhos_df_2 = pd.DataFrame(conformity_details_2)
    ganhos_df_2['dataset'] = dataset_name
    ganhos_df_2['method'] = 'WILCAR_Constrained_MSE'
    ganhos_df_2['n_neurons'] = best_n_2
    ganhos_df_2['timestamp'] = timestamp_2
    
    gains_path_2 = metrics_dir_2 / "gains_analysis.csv"
    ganhos_df_2.to_csv(gains_path_2, index=False)
    print(f"  ✓ Análise de ganhos: {gains_path_2.relative_to('..')}")

# === 4. SALVAR MELHOR MODELO ===
if best_model_2 is not None:
    model_path_2 = models_dir_2 / f"best_model_n{best_n_2}.pkl"
    with open(model_path_2, 'wb') as f:
        pickle.dump(best_model_2, f)
    print(f"  ✓ Modelo salvo: {model_path_2.relative_to('..')}")
    
    # Parâmetros do modelo (JSON legível)
    model_params_2 = {
        'n_neurons': best_n_2,
        'architecture': {
            'input_dim': n_inputs,
            'hidden_dim': best_n_2,
            'output_dim': 1,
            'activation': 'sigmoid',
            'constraints': 'gain_signs'
        },
        'weights_shapes': {
            'W1': list(best_model_2.param['W1'].shape),
            'b1': list(best_model_2.param['b1'].shape),
            'W2': list(best_model_2.param['W2'].shape),
            'b2': list(best_model_2.param['b2'].shape)
        },
        'performance': {
            'train_r2': float(best_metrics_2['Train R2']),
            'test_r2': float(best_metrics_2['Test R2']),
            'test_rmse': float(best_metrics_2['Test RMSE']),
            'conformity_rate': float(best_metrics_2['Conformity Rate'])
        },
        'optimization': {
            'method': 'SLSQP',
            'objective': 'MSE',
            'constraints': 'gain_signs',
            'margin': 0.05
        }
    }
    
    model_params_path_2 = models_dir_2 / "model_params.json"
    with open(model_params_path_2, 'w') as f:
        json.dump(model_params_2, f, indent=2)
    print(f"  ✓ Parâmetros do modelo: {model_params_path_2.relative_to('..')}")

# === 5. SALVAR FIGURAS ===
# Dashboard principal
dashboard_path_2 = figures_dir_2 / "performance_dashboard.png"
fig2.savefig(dashboard_path_2, dpi=300, bbox_inches='tight')
print(f"  ✓ Dashboard: {dashboard_path_2.relative_to('..')}")

# Alta resolução para artigo
dashboard_hires_path_2 = figures_dir_2 / "performance_dashboard_hires.pdf"
fig2.savefig(dashboard_hires_path_2, dpi=600, bbox_inches='tight', format='pdf')
print(f"  ✓ Dashboard HD (PDF): {dashboard_hires_path_2.relative_to('..')}")

# Gráfico específico de ganhos (para artigo)
if best_model_2 is not None:
    fig_gains_2, ax = plt.subplots(figsize=(10, 6))
    var_names = [f'Var {i+1}' for i in range(n_inputs)]
    x_pos = np.arange(n_inputs)
    
    colors = ['green' if detail['calculated_signal'] == detail['expected_signal'] or detail['expected_signal'] == 0
              else 'red' for detail in conformity_details_2]
    bars = ax.bar(x_pos, [d['calculated_gain'] for d in conformity_details_2], 
                  color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    expected_for_plot = [sinal if sinal != 0 else np.nan for sinal in expected_signals]
    ax.plot(x_pos, expected_for_plot, 'ko-', markersize=10, linewidth=2.5, 
            label='Expected Signal', zorder=5)
    
    ax.axhline(0, color='black', linestyle='-', linewidth=0.8)
    ax.set_xlabel('Input Variable', fontsize=12, fontweight='bold')
    ax.set_ylabel('Gain (∂ŷ/∂x)', fontsize=12, fontweight='bold')
    ax.set_title(f'Gain Conformity - WILCAR Constrained MSE (n={best_n_2}, TCS={tcs_final_2:.1%})', 
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(var_names)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    gains_fig_path_2 = figures_dir_2 / "gains_comparison.png"
    fig_gains_2.savefig(gains_fig_path_2, dpi=300, bbox_inches='tight')
    print(f"  ✓ Gráfico de ganhos: {gains_fig_path_2.relative_to('..')}")
    plt.close(fig_gains_2)

# === 6. CONFIG DA EXECUÇÃO ===
execution_config_2 = {
    'execution_info': {
        'timestamp': timestamp_2,
        'dataset': dataset_name,
        'method': 'WILCAR_Constrained_MSE'
    },
    'hyperparameters': {
        'max_neurons': MAX_NEURONS_2,
        'patience_constructive': PATIENCE_CONSTRUCTIVE_2,
        'delta_perturbation': DELTA_PERTURBATION_2,
        'optimization': {
            'method': 'SLSQP',
            'objective': 'MSE',
            'max_iter': 1000,
            'tolerance': 1e-6,
            'constraint_margin': 0.05
        }
    },
    'environment': {
        'seed': SEED,
    },
    'expected_signals': expected_signals
}

execution_config_path_2 = logs_dir_2 / "execution_config.json"
with open(execution_config_path_2, 'w') as f:
    json.dump(execution_config_2, f, indent=2)
print(f"  ✓ Execution config: {execution_config_path_2.relative_to('..')}")

# === 7. LOG DE TREINAMENTO ===
log_path_2 = logs_dir_2 / "training.log"
with open(log_path_2, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("WILCAR CONSTRAINED MSE TRAINING LOG\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Timestamp: {timestamp_2}\n")
    f.write(f"Dataset: {dataset_name}\n")
    f.write(f"Total time: {total_time_2:.2f}s ({total_time_2/60:.2f}min)\n")
    f.write(f"Models trained: {n_models_trained_2}/{MAX_NEURONS_2}\n")
    f.write(f"Convergence rate: {n_models_trained_2/len(optimization_success_2):.1%}\n\n")
    
    f.write("Best Model:\n")
    f.write(f"  Neurons: {best_n_2}\n")
    f.write(f"  Test R²: {best_metrics_2['Test R2']:.6f}\n")
    f.write(f"  Test RMSE: {best_metrics_2['Test RMSE']:.6f}\n")
    f.write(f"  Conformity Rate: {best_metrics_2['Conformity Rate']:.2%}\n\n")
    
    f.write("Iteration Details:\n")
    f.write("-" * 80 + "\n")
    for i, n in enumerate(neurons_2, 1):
        if optimization_success_2[i-1]:
            f.write(f"n={n:2d} | Train R²={r2_train_2[i-1]:.4f} | Test R²={r2_test_2[i-1]:.4f} | "
                    f"TCS={conformity_rates_2[i-1]:.2%} | Time={iteration_times_2[i-1]:.2f}s | SUCCESS\n")
        else:
            f.write(f"n={n:2d} | FAILED TO CONVERGE\n")

print(f"  ✓ Log de treinamento: {log_path_2.relative_to('..')}")

print("\n✅ Todos os resultados do Método 2 salvos com sucesso!")
print(f"📁 Diretório do experimento: {run_dir_2.relative_to('..')}")
print("=" * 80)

## Método 3  - RIXM
Modelo SFNN considerando inicialização de pesos a partir de uma inicialização randômica (Método de Xavier) e bias inicializados com valor nulo.

### Implementação do Método 3
Aproveita-se a classe definida para o Método 1.

In [ ]:
# Inicialização de variáveis para rastrear métricas de desempenho
mse_train_2 = []
r2_train_2 = []
rmse_train_2 = []
mse_test_2 = []
r2_test_2 = []
rmse_test_2 = []

# Inicialização de variáveis para rastrear o melhor modelo e suas métricas
best_metrics_2 = {
    'Train MSE': float('inf'),
    'Train RMSE': float('inf'),
    'Train R2': -float('inf'),
    'Test MSE': float('inf'),
    'Test RMSE': float('inf'),
    'Test R2': -float('inf'),
    'Best Neurons': 0
}
best_model_2 = None  # Para guardar o melhor modelo

# Marca o início da contagem do tempo
ini = time.time()

# Loop para treinar modelos com diferentes números de neurônios na camada oculta
for n in range(1, 51):
    if n == 1:
        model = Method1(train_inputs.T, train_targets.T, n, initial_weights=None)
    else:
        model = Method1(train_inputs.T, train_targets.T, n, previous_model=None)
    
    mse_train, rmse_train, r2_train, mse_test, rmse_test, r2_test = model.train_test()
    
    mse_train_2.append(mse_train)
    rmse_train_2.append(rmse_train)
    r2_train_2.append(r2_train)
    mse_test_2.append(mse_test)
    rmse_test_2.append(rmse_test)
    r2_test_2.append(r2_test)
    
    print(f"Nós Ocultos: {n} - MSE Treino = {mse_train}, RMSE Treino = {rmse_train}, R2 Treino = {r2_train}, "
          f"MSE Teste = {mse_test}, RMSE Teste = {rmse_test}, R2 Teste = {r2_test}")
    
    # Atualiza o melhor modelo se o atual for melhor que o registrado anteriormente
    if r2_test > best_metrics_2['Test R2']:
        best_metrics_2.update({
            'Train MSE': mse_train,
            'Train RMSE': rmse_train,
            'Train R2': r2_train,
            'Test MSE': mse_test,
            'Test RMSE': rmse_test,
            'Test R2': r2_test,
            'Best Neurons': n
        })
        best_model_2 = model  # Salva o modelo atual como o melhor modelo
    
    # Verifica os ganhos do modelo atual
    x_base = test_inputs.T
    delta = 0.1
    ganhos_calculados = []
    for i in range(n_inputs):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente
        y_base = model.predict(x_base)
        y_perturbed = model.predict(x_perturbed)
        ganho_i = (y_perturbed - y_base) / delta
        ganhos_calculados.append(np.mean(ganho_i))  # Tira a média de todos os ganhos calculados, por variável

    # Compara os ganhos calculados com os sinais de ganhos esperados
    for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados, expected_signals)):
        sinal_calculado = np.sign(ganho_calculado)
        
        if sinal_esperado == 0:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = Desconhecido")
        elif sinal_calculado == sinal_esperado:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Conforme]")
        else:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Divergente]")

    # Salva o modelo atual para a próxima iteração
    previous_model = model

# Marca o fim da contagem do tempo
end = time.time()

### Análise da Performance do Método 3

In [ ]:
print("Tempo de processamento (s):", end-ini)

print("Número ótimo de neurônios ocultos:", best_metrics_2['Best Neurons'])

print("MSE de treino do melhor modelo encontrado:", best_metrics_2['Train MSE'])
print("RMSE de treino do melhor modelo encontrado:", best_metrics_2['Train RMSE'])
#print("R2 de treino final com o melhor modelo:", best_metrics['Train R2'])

print("MSE de teste do melhor modelo:", best_metrics_2['Test MSE'])
print("RMSE de teste do melhor modelo:", best_metrics_2['Test RMSE'])
print("R2 de teste do melhor modelo:", best_metrics_2['Test R2'])

In [ ]:
# Define o conjunto de teste como referência para o cálculo dos ganhos
x_base = test_inputs.T

# Definindo delta (variação nas entradas)
delta = 0.1

# Calcula os ganhos para cada variável de entrada
ganhos_calculados = []
for i in range(n_inputs):
    x_perturbed = np.copy(x_base)
    x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente
    y_base = best_model_2.predict(x_base)
    y_perturbed = best_model_2.predict(x_perturbed)
    ganho_i = (y_perturbed - y_base) / delta
    ganhos_calculados.append(np.mean(ganho_i)) # Tira a média de todos os ganhos calculados, por variável

print('ANÁLISE DE GANHOS - MELHOR MODELO - RIXM')

# Compara os ganhos calculados com os sinais de ganhos esperados
for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados, expected_signals)):
    sinal_calculado = np.sign(ganho_calculado)
    
    if sinal_esperado == 0:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = Desconhecido")
    elif sinal_calculado == sinal_esperado:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Conforme]")
    else:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Divergente]")

In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_train_2)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do RMSE de treino:
plt.plot(rmse_train_2)
plt.ylabel('Train RMSE')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino:
#plt.plot(r2_train_2)
#plt.ylabel('Train R2')
#plt.xlabel('Hidden Nodes')
#plt.legend(['RIXM'])
#plt.show()

In [ ]:
# Criação e exibição de Gráfico do MSE de teste:
plt.plot(mse_test_2)
plt.ylabel('Test MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do RMSE de teste:
plt.plot(rmse_test_2)
plt.ylabel('Test RMSE')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de teste:
plt.plot(r2_test_2)
plt.ylabel('Test R2')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

### Salvando Métricas do Método 3

In [ ]:
metrics_2 = {
    'mse_train': mse_train_2,
    'rmse_train': rmse_train_2,
    'r2_train': r2_train_2,
    'mse_test': mse_test_2,
    'rmse_test': rmse_test_2,
    'r2_test': r2_test_2
}

In [ ]:
save_metrics(metrics_2, 'RIXM')

## Método 4 - RIXM Com Restrições dos Sinais dos Ganhos - MSE

### Implementação do Método 4 (RIXM) - Com Restrições dos Sinais dos Ganhos - MSE

In [ ]:
class Method2_CONS:
    def __init__(self, input_size, output_size=1):
        self.input_size = input_size
        self.output_size = output_size

    @staticmethod
    def sigmoid(x):
        clipped_x = np.clip(x, -500, 500)  # Limita valores para evitar extremos em exp
        return np.where(clipped_x >= 0, 
                        1 / (1 + np.exp(-clipped_x)), 
                        np.exp(clipped_x) / (1 + np.exp(clipped_x)))

    def initialize_weights(self, n, x_train, expected_signals):
        """
        Inicializa os pesos usando a inicialização Xavier e ajusta até que os ganhos satisfaçam as restrições.
        """
        init_range = np.sqrt(6 / (self.input_size + n))
        while True:
            W1 = np.random.uniform(-init_range, init_range, (n, self.input_size))
            b1 = np.zeros((n, 1))
            W2 = np.random.uniform(-init_range, init_range, (1, n))
            b2 = np.zeros((1, 1))
            params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

            # Verifica se os ganhos iniciais satisfazem as restrições
            all_gains_match = True
            for i, expected_gain in enumerate(expected_signals):
                if expected_gain != 0:
                    calculated_gain = self.calculate_gain(params, x_train, i)
                    gain_match = np.sign(calculated_gain) == expected_gain
                    all_gains_match &= gain_match
                    if not gain_match:
                        break

            if all_gains_match:
                break

        return params

    def predict(self, x, params):
        Z1 = np.dot(params['W1'], x) + params['b1']
        A1 = self.sigmoid(Z1)
        Z2 = np.dot(params['W2'], A1) + params['b2']
        A2 = self.sigmoid(Z2)
        return A2

    def objective(self, params, x, y):
        A2 = self.predict(x, params)
        return mse(y, A2.flatten())

    def calculate_gain(self, params, x, var_index, delta=0.1):
        x_perturbed = np.copy(x)
        x_perturbed[var_index, :] += delta
        y_base = self.predict(x, params)
        y_perturbed = self.predict(x_perturbed, params)
        gain = np.mean((y_perturbed - y_base) / delta, axis=1)
        return gain[0]

    def gain_constraint(self, params, x, var_index, expected_gain):
        actual_gain = self.calculate_gain(params, x, var_index)
        if expected_gain == 1:
            return actual_gain - 0.1
        elif expected_gain == -1:
            return -actual_gain - 0.1
        return 0

    def train_network(self, train_inputs, train_targets, test_inputs, test_targets, expected_signals):
        x_train = train_inputs.T
        y_train = train_targets.T
        x_test = test_inputs.T
        y_test = test_targets.T

        mse_train = []
        rmse_train = []
        mse_test = []
        rmse_test = []
        r2_train = []
        r2_test = []
        gain_matches = []
        best_metrics = {
            'Train MSE': float('inf'),
            'Train RMSE': float('inf'),
            'Train R2': -float('inf'),
            'Test MSE': float('inf'),
            'Test RMSE': float('inf'),
            'Test R2': -float('inf')
        }
        best_params = None

        for n in range(1, 51):
            print(f"Treinando com {n} neurônios...")
            params = self.initialize_weights(n, x_train, expected_signals)
            W0 = np.concatenate([params['W1'].flatten(), params['b1'].flatten(), params['W2'].flatten(), params['b2'].flatten()])

            constraints = [{'type': 'ineq', 'fun': lambda W, x=x_train, n=n, i=i, sig=expected_signals[i]: self.gain_constraint(self.param_vector_to_dict(W, n), x, i, sig),
                            'args': []} for i in range(self.input_size) if expected_signals[i] != 0]

            result = minimize(lambda W: self.objective(self.param_vector_to_dict(W, n), x_train, y_train),
                              W0, method='SLSQP', constraints=constraints,
                              options={'disp': True, 'maxiter': 1000, 'ftol': 1e-05})

            if result.success:
                optimized_params = self.param_vector_to_dict(result.x, n)

                current_mse = self.objective(optimized_params, x_train, y_train)
                mse_train.append(current_mse)
                current_rmse = np.sqrt(current_mse)
                rmse_train.append(current_rmse)
                current_r2_train = r2_score(y_train.flatten(), self.predict(x_train, optimized_params).flatten())
                r2_train.append(current_r2_train)

                y_test_pred = self.predict(x_test, optimized_params)
                test_mse = mse(y_test, y_test_pred.flatten())
                mse_test.append(test_mse)
                test_rmse = np.sqrt(test_mse)
                rmse_test.append(test_rmse)
                current_r2_test = r2_score(y_test, y_test_pred.flatten())
                r2_test.append(current_r2_test)

                all_gains_match = True
                for i, expected_gain in enumerate(expected_signals):
                    calculated_gain = self.calculate_gain(optimized_params, x_test, i)
                    gain_match = np.sign(calculated_gain) == expected_gain
                    all_gains_match &= gain_match
                
                gain_matches.append(all_gains_match)

                if current_r2_test > best_metrics['Test R2']:
                    best_metrics.update({
                        'Train MSE': current_mse,
                        'Train RMSE': current_rmse,
                        'Train R2': current_r2_train,
                        'Test MSE': test_mse,
                        'Test RMSE': test_rmse,
                        'Test R2': current_r2_test
                    })
                    best_params = optimized_params

                print(f"Treinamento com {n} neurônios concluído, R2 de Teste: {current_r2_test}, Correspondência de Ganhos: {all_gains_match}")
            else:
                print(f"Treinamento com {n} neurônios falhou na otimização.")
                break

            if not all_gains_match:
                print(f"Treinamento interrompido em {n} neurônios. Problema de correspondência de ganhos.")
                break

        # Análise final dos ganhos para o melhor modelo
        final_gain_matches = []
        if best_params:
            for i, expected_gain in enumerate(expected_signals):
                final_calculated_gain = self.calculate_gain(best_params, x_test, i)
                final_gain_matches.append(np.sign(final_calculated_gain) == expected_gain)

        return mse_train, rmse_train, mse_test, rmse_test, r2_train, r2_test, best_metrics, best_params, gain_matches

    def param_vector_to_dict(self, vector, n):
        n_W1 = n * self.input_size
        n_b1 = n
        n_W2 = n
        n_b2 = 1  # Porque W2 tem apenas uma linha e b2 é um único termo de bias para a saída

        W1 = vector[:n_W1].reshape(n, self.input_size)
        b1 = vector[n_W1:n_W1 + n_b1].reshape(n, 1)
        W2 = vector[n_W1 + n_b1:n_W1 + n_b1 + n_W2].reshape(1, n)
        b2 = vector[-n_b2:].reshape(1, 1)

        return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

In [ ]:
ini = time.time()
nn = Method2_CONS(input_size=input_size)
mse_train, rmse_train, mse_test, rmse_test, r2_train, r2_test, best_metrics, best_params, gain_matches = nn.train_network(train_inputs, train_targets, test_inputs, test_targets, expected_signals)
fim = time.time()

### Análise da Performance do Método 4

In [ ]:
# Guardando Métricas:
best_metrics2_cons_mse = best_metrics
mse_train2_cons_mse = mse_train
mse_test2_cons_mse = mse_test
r2_train2_cons_mse = r2_train
r2_test2_cons_mse = r2_test
rmse_train2_cons_mse = rmse_train
rmse_test2_cons_mse = rmse_test
best_model2_cons_params_mse = best_params

In [ ]:
print("Tempo de processamento (s):", fim - ini)

print("Número ótimo de neurônios ocultos:", best_metrics2_cons_mse['Best Neurons'])

print("MSE de treino do melhor modelo encontrado:", best_metrics2_cons_mse['Train MSE'])
print("RMSE de treino do melhor modelo encontrado:", best_metrics2_cons_mse['Train RMSE'])
print("R2 de treino do melhor modelo encontrado:", best_metrics2_cons_mse['Train R2'])

print("MSE de teste do melhor modelo:", best_metrics2_cons_mse['Test MSE'])
print("RMSE de teste do melhor modelo:", best_metrics2_cons_mse['Test RMSE'])
print("R2 de teste do melhor modelo:", best_metrics2_cons_mse['Test R2'])

In [ ]:
# Criação e exibição de gráfico do MSE de treino:
plt.plot(mse_train2_cons_mse)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Noddes')
plt.legend(['C-RIXM (MSE)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do RMSE de treino:
plt.plot(rmse_train2_cons_mse)
plt.ylabel('Train RMSE')
plt.xlabel('Hidden Noddes')
plt.legend(['C-RIXM (MSE)'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino:
#plt.plot(r2_train1_cons)
#plt.ylabel('Train R2')
#plt.xlabel('Hidden Nodes')
#plt.legend(['Constrained WILCAR'])
#plt.show()

In [ ]:
# Criação e exibição de gráfico do MSE de teste:
plt.plot(mse_test2_cons_mse)
plt.ylabel('Test MSE')
plt.xlabel('Hidden Noddes')
plt.legend(['C-RIXM (MSE)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do RMSE de teste:
plt.plot(rmse_test2_cons_mse)
plt.ylabel('Test RMSE')
plt.xlabel('Hidden Noddes')
plt.legend(['C-RIXM (MSE)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do R2 de teste:
plt.plot(r2_test2_cons_mse)
plt.ylabel('Test R2')
plt.xlabel('Hidden Noddes')
plt.legend(['C-RIXM (MSE)'])
plt.show()

### Salvando Métricas do Método 4

In [ ]:
# Salvando Métricas do Método 2 com Restrições dos Sinais dos Ganhos (Constrained RIXM - MSE)
metrics_2_cons_rmse = {
    'mse_train': mse_train2_cons_mse,
    'rmse_train': rmse_train2_cons_mse,
    'r2_train': r2_train2_cons_mse,
    'mse_test': mse_test2_cons_mse,
    'rmse_test': rmse_test2_cons_mse,
    'r2_test': r2_test2_cons_mse
}

In [ ]:
save_metrics(metrics_2_cons_rmse, 'C_RIXM_MSE')

## Método 5 - ELM
Modelo ELM considerando os pesos iniciais e bias do neurônio oculto e do neurônio de saída obtidos a partir do método de inicialização randômica clássica, desconsiderando os pesos já aprendidos.

### Definição do Método 5

In [ ]:
# Função de ativação ReLU:
def relu(x):
    return np.maximum(x, 0)

# Função para calcular nós ocultos:
def hidden_nodes(X, input_weights, biases):
    G = np.dot(X, input_weights)
    G = G + biases
    H = relu(G)
    return H

# Função de previsão:
def predict_test(X, output_weights, input_weights, biases):
    out = hidden_nodes(X, input_weights, biases)
    out = np.dot(out, output_weights)
    return out

# Função para cálculo de ganhos:
def calculate_gains_elm(input_weights, biases, output_weights, x_base, delta=0.1):
    n_inputs = x_base.shape[0]
    gains_calculated = []

    for i in range(n_inputs):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente

        # Previsão base
        out_base = hidden_nodes(x_base.T, input_weights, biases)
        y_base = np.dot(out_base, output_weights)

        # Previsão perturbada
        out_perturbed = hidden_nodes(x_perturbed.T, input_weights, biases)
        y_perturbed = np.dot(out_perturbed, output_weights)

        gain_i = (y_perturbed - y_base) / delta
        gains_calculated.append(np.mean(gain_i))  # Calcula a média dos ganhos

    return gains_calculated

### Implementação do Método 5

In [ ]:
# Inicializando contagem de tempo:
ini = time.time()

# Abrindo listas para R2 (treino e teste), MSE e RMSE:
mse_train_3 = []
rmse_train_3 = []
mse_test_3 = []
rmse_test_3 = []
r2_train_3 = []
r2_test_3 = []

# Abrindo lista para registro de pesos dos modelos:
params_models_3 = []

# Inicializando o dicionário para rastrear as melhores métricas:
best_metrics3_elm = {
    'Train MSE': float('inf'),
    'Train RMSE': float('inf'),
    'Train R2': -float('inf'),
    'Test MSE': float('inf'),
    'Test RMSE': float('inf'),
    'Test R2': -float('inf'),
    'Best Neurons': 0
}

# Definindo tamanho do conjunto de entrada:
input_size = train_inputs.shape[1]

# Inicializando loop para neurônios ocultos:
for n in range(1, 51):
    # Treino:
    input_weights = np.random.uniform(-1, 1, (input_size, n))
    biases = np.zeros(n)
    print("Iniciando rede com", n, "neurônios ocultos")

    out = hidden_nodes(train_inputs, input_weights, biases)
    output_weights = np.dot(pinv(out), train_targets)
    out = np.dot(out, output_weights)
    actual = train_targets

    train_mse = mse(actual, out)
    train_rmse = np.sqrt(train_mse) 
    train_r2 = r2_score(actual, out)

    mse_train_3.append(train_mse)
    rmse_train_3.append(train_rmse)
    r2_train_3.append(train_r2)

    # Registrando pesos já aprendidos:
    params_models_3.append((input_weights, biases, output_weights))

    # Teste:
    test_predict = predict_test(test_inputs, output_weights, input_weights, biases)
    actual = test_targets

    test_mse = mse(actual, test_predict)
    test_rmse = np.sqrt(test_mse)  
    test_r2 = r2_score(actual, test_predict)

    mse_test_3.append(test_mse)
    rmse_test_3.append(test_rmse)
    r2_test_3.append(test_r2)

    if test_r2 > best_metrics3_elm['Test R2']:
        best_metrics3_elm.update({
            'Train MSE': train_mse,
            'Train RMSE': train_rmse,
            'Train R2': train_r2,
            'Test MSE': test_mse,
            'Test RMSE': test_rmse,
            'Test R2': test_r2,
            'Best Neurons': n
        })

    x_base = test_inputs.T

    # Calcula ganhos:
    delta = 0.1
    gains_calculated = calculate_gains_elm(input_weights, biases, output_weights, x_base, delta)

    # Compara ganhos calculados com sinais esperados:
    for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
        calculated_signal = np.sign(gain_calculated)
        if expected_signal == 0:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
        elif calculated_signal == expected_signal:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
        else:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")

fim = time.time()

### Avaliação do Método 5

In [ ]:
# Exibição do tempo de processamento:
print("Tempo de processamento:", fim - ini)

# Exibição das melhores métricas:
print("Número ótimo de neurônios ocultos:", best_metrics3_elm['Best Neurons'])
print("MSE de treino do melhor modelo encontrado:", best_metrics3_elm['Train MSE'])
print("RMSE de treino do melhor modelo encontrado:", best_metrics3_elm['Train RMSE'])
print("R2 de treino do melhor modelo encontrado:", best_metrics3_elm['Train R2'])
print("MSE de teste do melhor modelo:", best_metrics3_elm['Test MSE'])
print("RMSE de teste do melhor modelo:", best_metrics3_elm['Test RMSE'])
print("R2 de teste do melhor modelo:", best_metrics3_elm['Test R2'])

In [ ]:
# Verificação de ganhos no melhor modelo (usando o modelo com melhor R2 no teste):
best_model_idx = np.argmax(r2_test_3)
best_input_weights, best_biases, best_output_weights = params_models_3[best_model_idx]

x_base = test_inputs.T

# Calcula ganhos:
delta = 0.1
gains_calculated = calculate_gains_elm(best_input_weights, best_biases, best_output_weights, x_base, delta)
print('ANÁLISE DE GANHOS - MELHOR MODELO - ELM')
# Compara ganhos calculados com sinais esperados:
for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
    calculated_signal = np.sign(gain_calculated)
    if expected_signal == 0:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
    elif calculated_signal == expected_signal:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
    else:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")

In [ ]:
# Gráfico do MSE de treino:
plt.plot(mse_train_3)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Gráfico do RMSE de treino:
plt.plot(rmse_train_3)
plt.ylabel('Train RMSE')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Gráfico do MSE de teste:
plt.plot(mse_test_3)
plt.ylabel('Test MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Gráfico do RMSE de teste:
plt.plot(rmse_test_3)
plt.ylabel('Test RMSE')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Gráfico do R2 de teste:
plt.plot(r2_test_3)
plt.ylabel('Test R2')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

### Salvando Métricas do Método 5 (ELM)

In [ ]:
metrics_3 = {
    'mse_train': mse_train_3,
    'rmse_train': rmse_train_3,
    'r2_train': r2_train_3,
    'mse_test': mse_test_3,
    'rmse_test': rmse_test_3,
    'r2_test': r2_test_3
}

In [ ]:
save_metrics(metrics_3, 'ELM')

## Método 6 - ELM Com Restrições nos Sinais dos Ganhos - MSE

### Definição do Método 6

In [ ]:
# Função de ativação ReLU
def relu(x):
    return np.maximum(x, 0)

# Função para calcular nós ocultos
def hidden_nodes(X, input_weights, biases):
    G = np.dot(X, input_weights)
    G = G + biases
    H = relu(G)
    return H

# Função de previsão
def predict_test(X, output_weights, input_weights, biases):
    out = hidden_nodes(X, input_weights, biases)
    out = np.dot(out, output_weights)
    return out

# Função de restrição de ganho
def gain_constraint_elm(x, idx, x_base, expected_signals, delta=0.1, input_size=6, hidden_neurons=1):
    input_weights = x[:input_size * hidden_neurons].reshape(input_size, hidden_neurons)
    biases = x[input_size * hidden_neurons: input_size * hidden_neurons + hidden_neurons]
    output_weights = x[input_size * hidden_neurons + hidden_neurons:]
    
    x_var = x_base.copy()
    x_var[idx] += delta
    y_base = predict_test(x_base.T, output_weights, input_weights, biases)
    new_y = predict_test(x_var.T, output_weights, input_weights, biases)
    gain = (new_y - y_base) / delta
    
    if expected_signals[idx] != 0:
        return expected_signals[idx] * gain - 0.01
    else:
        return 1

# Função para cálculo de ganhos:
def calculate_gains_elm(input_weights, biases, output_weights, x_base, delta=0.1):
    n_inputs = x_base.shape[0]
    gains_calculated = []

    for i in range(n_inputs):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente

        # Previsão base
        out_base = hidden_nodes(x_base.T, input_weights, biases)
        y_base = np.dot(out_base, output_weights)

        # Previsão perturbada
        out_perturbed = hidden_nodes(x_perturbed.T, input_weights, biases)
        y_perturbed = np.dot(out_perturbed, output_weights)

        gain_i = (y_perturbed - y_base) / delta
        gains_calculated.append(np.mean(gain_i))  # Calcula a média dos ganhos

    return gains_calculated

def train_elm_with_constraints(W0, cons, train_inputs, train_targets, hidden_neurons, input_size):
    def objective_function_elm(W):
        input_weights = W[:input_size * hidden_neurons].reshape(input_size, hidden_neurons)
        biases = W[input_size * hidden_neurons: input_size * hidden_neurons + hidden_neurons]
        output_weights = W[input_size * hidden_neurons + hidden_neurons:]
        H = hidden_nodes(train_inputs, input_weights, biases)
        predictions = np.dot(H, output_weights)
        regularization = 0.01 * (np.sum(np.square(input_weights)) + np.sum(np.square(output_weights)))
        return mse(train_targets, predictions) + regularization

    solution = minimize(objective_function_elm, W0, method='SLSQP', constraints=cons, options={'disp': True, 'maxiter': 500, 'ftol': 1e-06})
    return solution['x']

def initialize_weights_with_constraints(input_size, hidden_neurons, expected_signals, x_base, delta=0.1):
    def xavier_initialization(input_size, hidden_neurons):
        limit = np.sqrt(6 / (input_size + hidden_neurons))
        W0 = np.random.uniform(-limit, limit, (input_size * hidden_neurons + hidden_neurons + hidden_neurons))
        return W0

    first_attempt = True
    while True:
        W0 = xavier_initialization(input_size, hidden_neurons)
        input_weights = W0[:input_size * hidden_neurons].reshape(input_size, hidden_neurons)
        biases = W0[input_size * hidden_neurons: input_size * hidden_neurons + hidden_neurons]
        output_weights = W0[input_size * hidden_neurons + hidden_neurons:]

        gains_calculated = calculate_gains_elm(input_weights, biases, output_weights, x_base, delta)

        all_gains_match = True
        for i, expected_signal in enumerate(expected_signals):
            if expected_signal != 0:
                calculated_signal = np.sign(gains_calculated[i])
                if calculated_signal != expected_signal:
                    all_gains_match = False
                    break
        
        if all_gains_match:
            return W0

        if first_attempt:
            print("Pesos iniciais não satisfazem as restrições, tentando novamente.")
            first_attempt = False

### Implementação do Método 6

In [ ]:
# Inicializar variáveis
mse_train_3_cons = []
mse_test_3_cons = []
r2_train_3_cons = []
r2_test_3_cons = []
params_models_3_cons = []

# Definir tamanho da entrada
input_size = train_inputs.shape[1]

# Inicializar temporizador
ini = time.time()

# Loop sobre neurônios ocultos
for n in range(1, 51):
    print(f"Iniciando rede com {n} neurônios ocultos")

    x_base = np.mean(test_inputs, axis=0).reshape(input_size, 1)
    
    while True:
        W0 = initialize_weights_with_constraints(input_size, n, expected_signals, x_base)
        cons = [{'type': 'ineq', 'fun': lambda W, idx=i: gain_constraint_elm(W, idx, x_base, expected_signals, delta=0.1, input_size=input_size, hidden_neurons=n)} for i in range(input_size)]
        
        W_opt = train_elm_with_constraints(W0, cons, train_inputs, train_targets, n, input_size)
        
        input_weights = W_opt[:input_size * n].reshape(input_size, n)
        biases = W_opt[input_size * n:input_size * n + n]
        output_weights = W_opt[input_size * n + n:]
        
        # Calcula ganhos:
        gains_calculated = calculate_gains_elm(input_weights, biases, output_weights, test_inputs.T)

        # Verifica correspondência de ganhos
        all_gains_match = True
        for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
            calculated_signal = np.sign(gain_calculated)
            if expected_signal == 0:
                print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
            elif calculated_signal == expected_signal:
                print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
            else:
                print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")
                all_gains_match = False

        if all_gains_match:
            break
        else:
            print("Pesos após adição do neurônio não satisfazem as restrições, recalculando...")

    # Treinar
    H = hidden_nodes(train_inputs, input_weights, biases)
    out_train = np.dot(H, output_weights)
    mse_train_3_cons.append(mse(train_targets, out_train))
    r2_train_3_cons.append(r2_score(train_targets, out_train))
    params_models_3_cons.append((input_weights, biases, output_weights))
    
    # Testar
    out_test = predict_test(test_inputs, output_weights, input_weights, biases)
    mse_test_3_cons.append(mse(test_targets, out_test))
    r2_test_3_cons.append(r2_score(test_targets, out_test))

    # Calcula ganhos:
    gains_calculated = calculate_gains_elm(input_weights, biases, output_weights, test_inputs.T)

    # Verifica correspondência de ganhos
    all_gains_match = True
    for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
        calculated_signal = np.sign(gain_calculated)
        if expected_signal == 0:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
        elif calculated_signal == expected_signal:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
        else:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")
            all_gains_match = False

    if not all_gains_match:
        print(f"Treinamento interrompido em {n} neurônios. Problema de correspondência de ganhos.")
        break

# Análise da Performance do Método 3:
fim = time.time()
print("Tempo de processamento:", fim-ini)

### Avaliação do Método 6

In [ ]:
# Verificação de ganhos no melhor modelo
best_model_idx = np.argmax(r2_test_3_cons)
best_input_weights, best_biases, best_output_weights = params_models_3_cons[best_model_idx]

x_base = test_inputs.T

# Calcular ganhos
delta = 0.1
gains_calculated = calculate_gains_elm(best_input_weights, best_biases, best_output_weights, x_base, delta)

# Comparar ganhos calculados com sinais esperados
for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
    calculated_signal = np.sign(gain_calculated)
    if expected_signal == 0:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
    elif calculated_signal == expected_signal:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
    else:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")


In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_train_3_cons)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['Constrained ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_test_3_cons)
plt.ylabel('Test MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['Constrained ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de teste:
plt.plot(r2_test_3_cons)
plt.ylabel('Test R2')
plt.xlabel('Hidden Nodes')
plt.legend(['Constrained ELM'])
plt.show()

In [ ]:
print("R2 máximo de teste obtido:", max(r2_test_3_cons))
print("Número ótimo de neurônios ocultos:", (r2_test_3_cons.index(max(r2_test_3_cons))) + 1)

### Salvando Métricas do Método 6

In [ ]:
metrics_3_cons = {
    'mse_train': mse_train_3_cons,
    'mse_test': mse_test_3_cons,
    'r2_test': r2_test_3_cons
}

In [ ]:
save_metrics(metrics_3_cons, 'C_ELM_MSE')

## GRÁFICOS DOS MÉTODOS EM CONJUNTO

In [ ]:
import matplotlib
matplotlib.rc('xtick', labelsize=10)
matplotlib.rc('ytick', labelsize=10)

### MSE de Treino

In [ ]:
# Gráfico em conjunto do MSE de treino (até 250 neurônios ocultos)
x = np.arange(1, 251)
y1 = mse_train_1
y2 = mse_train1_cons_mse
y3 = mse_train1_cons_rmse
y4 = mse_train_2
y5 = mse_train2_cons_mse
y6 = mse_train2_cons_rmse
y7 = mse_train_3
y8 = mse_train_3_cons

# Inicializa a figura e os eixos
fig, ax = plt.subplots(1, figsize=(12, 8), dpi=100)

# Plota todas as linhas no mesmo gráfico, atribuindo um rótulo para cada uma para ser exibido na legenda
ax.plot(x, y1, color="green", label="WILCAR")
ax.plot(x, y2, label="C-WILCAR (MSE)")
ax.plot(x, y3, label="C-WILCAR (RMSE)")
ax.plot(x, y4, color="red", label="RIXM")
ax.plot(x, y5, label="C-RIXM (MSE)")
ax.plot(x, y6, label="C-RIXM (RMSE)")
ax.plot(x, y7, color="blue", label="ELM")
ax.plot(x, y8, label="C-ELM")

# Adiciona uma legenda e a posiciona no canto inferior direito (sem caixa)
plt.legend()

# Nome dos eixos
plt.ylabel('Train MSE', fontsize=12)
plt.xlabel('Hidden Noddes', fontsize=12)
# Para ampliar o gráfico
# plt.xlim(0, 50)
plt.show()

### R2 de Teste

In [ ]:
# Gráfico em conjunto do R2 de teste (até 250 neurônios ocultos)
x = np.arange(1, 251)
y1 = r2_test_1
y2 = r2_test1_cons_mse
y3 = r2_test1_cons_rmse
y4 = r2_test_2
y5 = r2_test2_cons_mse
y6 = r2_test2_cons_rmse
y7 = r2_test_3
y8 = r2_test_3_cons

# Inicializa a figura e os eixos
fig, ax = plt.subplots(1, figsize=(12, 8), dpi=100)

# Plota todas as linhas no mesmo gráfico, atribuindo um rótulo para cada uma para ser exibido na legenda
ax.plot(x, y1, color="green", label="WILCAR")
ax.plot(x, y2, label="C-WILCAR (MSE)")
ax.plot(x, y3, label="C-WILCAR (RMSE)")
ax.plot(x, y4, color="red", label="RIXM")
ax.plot(x, y5, label="C-RIXM (MSE)")
ax.plot(x, y6, label="C-RIXM (RMSE)")
ax.plot(x, y7, color="blue", label="ELM")
ax.plot(x, y8, label="C-ELM")

# Adiciona uma legenda e a posiciona no canto inferior direito (sem caixa)
plt.legend()

# Nome dos eixos
plt.ylabel('Test R2', fontsize=12)
plt.xlabel('Hidden Noddes', fontsize=12)
plt.ylim(0, 1)
plt.show()

## FIM DO CÓDIGO